# CLOVER

## Imports

In [ ]:
from pathlib import Path
from __future__ import annotations
import os
import re
import gzip
import time
import requests
import numpy as np
import pandas as pd
from collections import defaultdict
from io import StringIO
from IPython.display import display
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt


import csv
import warnings
from pathlib import Path
from urllib.parse import urlencode

START = time.perf_counter()
def log(message):
    print(f"[{time.perf_counter()-START:8.1f}s] {message}", flush=True)

pd.set_option("display.max_columns", 80)

## 1. Configuration

In [ ]:
ROOT = Path("/Users/sc/nlp_4_ctd_code/data_json_clover")
CTD_DIR = ROOT / "CTD_data"
INPUT_DATA_PATH = ROOT / "BedrockInput"
OUTPUT_DATA_PATH = ROOT / "BedrockOutput"
REFINED_DIR = OUTPUT_DATA_PATH / "refined_combined"

INPUT_CSV = INPUT_DATA_PATH / "input_combined.csv"
INPUT_VAL_CSV = INPUT_DATA_PATH / "input_val_combined.csv"
VALIDATION_CSV = OUTPUT_DATA_PATH / "validation_combined.csv"
VALIDATION_COPY_CSV = OUTPUT_DATA_PATH / "validation_copy_combined.csv"

SAVE_DIR = ROOT / "CLOVER_output_files" / "ctd_matching_combined"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

if not REFINED_DIR.is_dir():
    raise FileNotFoundError(f"Missing refined_combined folder: {REFINED_DIR}")
print("Model input folder:", REFINED_DIR)
print("Save location:", SAVE_DIR)


## 2. Gene normalization


In [ ]:
from __future__ import annotations

import csv
import gzip
import re
import warnings
from collections import defaultdict
from io import StringIO
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
import requests

SPECIES_TAXIDS = {"hsa": 9606, "rno": 10116, "mmu": 10090}
TAXID_SPECIES = {v: k for k, v in SPECIES_TAXIDS.items()}
NCBI_FILES = {9606: "Homo_sapiens.gene_info.gz", 10116: "Rattus_norvegicus.gene_info.gz", 10090: "Mus_musculus.gene_info.gz"}
NCBI_ROOT = "https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/"
HGNC_URL = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
UNIPROT_ENDPOINT = "https://rest.uniprot.org/uniprotkb/stream"
CACHE_DIR = Path("./mirgenedb_normalization_reports/reference_data")

# Biological aliases are NOT hardcoded.
# Gene aliases come from NCBI Gene, HGNC, and UniProt.
# Human/rat/mouse taxonomy codes and data URLs remain configuration constants.

_REFERENCE_CACHE.clear()
_SESSION = requests.Session()
_REFERENCE_CACHE = {}


def _clean(v):
    if v is None:
        return ""
    try:
        if bool(pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    s = str(v).strip()
    return "" if s.lower() in {"", "nan", "none", "<na>", "null", "-"} else s


def _key(v):
    return re.sub(r"[^a-z0-9]", "", _clean(v).casefold())


def _id(v):
    return re.sub(r"\.0$", "", _clean(v))


def _species_from_text(v):
    m = re.match(r"^(hsa|rno|mmu)[-_]", _clean(v), re.I)
    return m.group(1).lower() if m else ""


def _download(url, path, *, min_bytes=20):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size >= min_bytes:
        return path
    temp = path.with_suffix(path.suffix + ".tmp")
    try:
        with _SESSION.get(url, stream=True, timeout=(20, 120), headers={"User-Agent": "CLOVER-normalizer/4.0"}) as r:
            r.raise_for_status()
            with temp.open("wb") as out:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        out.write(chunk)
        if temp.stat().st_size < min_bytes:
            raise ValueError(f"Empty or truncated download: {url}")
        if path.suffix == ".gz":
            with gzip.open(temp, "rb") as f:
                f.read(32)
        temp.replace(path)
        return path
    finally:
        temp.unlink(missing_ok=True)


def _put(mapping, taxid, label, gid):
    k = _key(label)
    if k and gid:
        mapping[(taxid, k)].add(gid)


def _load_ncbi(cache_dir):
    by_id = {}

In [ ]:
from __future__ import annotations

import csv
import gzip
import re
import warnings
from collections import defaultdict
from io import StringIO
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
import requests

SPECIES_TAXIDS = {"hsa": 9606, "rno": 10116, "mmu": 10090}
TAXID_SPECIES = {v: k for k, v in SPECIES_TAXIDS.items()}
NCBI_FILES = {9606: "Homo_sapiens.gene_info.gz", 10116: "Rattus_norvegicus.gene_info.gz", 10090: "Mus_musculus.gene_info.gz"}
NCBI_ROOT = "https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/"
HGNC_URL = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
UNIPROT_ENDPOINT = "https://rest.uniprot.org/uniprotkb/stream"
CACHE_DIR = Path("./mirgenedb_normalization_reports/reference_data")

# Biological aliases are NOT hardcoded.
# Gene aliases come from NCBI Gene, HGNC, and UniProt.
# Human/rat/mouse taxonomy codes and data URLs remain configuration constants.

_SESSION = requests.Session()
_REFERENCE_CACHE = {}


def _clean(v):
    if v is None:
        return ""
    try:
        if bool(pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    s = str(v).strip()
    return "" if s.lower() in {"", "nan", "none", "<na>", "null", "-"} else s


# Preserve Greek-letter meaning during gene-name normalization
GREEK = str.maketrans({
    "α": "alpha",
    "β": "beta",
    "γ": "gamma",
    "δ": "delta",
    "κ": "kappa",
})

def _key(v):
    s = _clean(v).casefold().translate(GREEK)
    return re.sub(r"[^a-z0-9]", "", s)


def _id(v):
    return re.sub(r"\.0$", "", _clean(v))


def _species_from_text(v):
    m = re.match(r"^(hsa|rno|mmu)[-_]", _clean(v), re.I)
    return m.group(1).lower() if m else ""


def _download(url, path, *, min_bytes=20):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size >= min_bytes:
        return path
    temp = path.with_suffix(path.suffix + ".tmp")
    try:
        with _SESSION.get(url, stream=True, timeout=(20, 120), headers={"User-Agent": "CLOVER-normalizer/4.0"}) as r:
            r.raise_for_status()
            with temp.open("wb") as out:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        out.write(chunk)
        if temp.stat().st_size < min_bytes:
            raise ValueError(f"Empty or truncated download: {url}")
        if path.suffix == ".gz":
            with gzip.open(temp, "rb") as f:
                f.read(32)
        temp.replace(path)
        return path
    finally:
        temp.unlink(missing_ok=True)


def _put(mapping, taxid, label, gid):
    k = _key(label)
    if k and gid:
        mapping[(taxid, k)].add(gid)


def _load_ncbi(cache_dir):
    by_id = {}
    official = defaultdict(set)
    synonyms = defaultdict(set)
    descriptions = defaultdict(set)
    for tid, fn in NCBI_FILES.items():
        path = _download(NCBI_ROOT + fn, cache_dir / fn)
        with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
            header = next(f).lstrip("#").rstrip("\n").split("\t")
            columns = {name: i for i, name in enumerate(header)}
            required = {"tax_id", "GeneID", "Symbol", "Synonyms"}
            if not required <= columns.keys():
                raise ValueError(f"Invalid NCBI gene_info header: {path}")
            for line in f:
                parts = line.rstrip("\n").split("\t")
                if len(parts) < len(header) or parts[columns["tax_id"]] != str(tid):
                    continue
                gid, sym = _id(parts[columns["GeneID"]]), _clean(parts[columns["Symbol"]])
                if not gid or not sym:
                    continue
                desc = _clean(parts[columns["description"]]) if "description" in columns else ""
                by_id[(tid, gid)] = (sym, desc or sym)
                _put(official, tid, sym, gid)
                _put(descriptions, tid, desc, gid)
                for synonym in parts[columns["Synonyms"]].split("|"):
                    _put(synonyms, tid, synonym, gid)
    return by_id, official, synonyms, descriptions


def _load_hgnc(cache_dir, by_id):
    aliases = defaultdict(set)
    descriptions = defaultdict(set)
    path = _download(HGNC_URL, cache_dir / "hgnc_complete_set.txt", min_bytes=200)
    with path.open(encoding="utf-8", errors="replace", newline="") as f:
        for row in csv.DictReader(f, delimiter="\t"):
            if _clean(row.get("status")).lower() not in {"", "approved"}:
                continue
            gid = _id(row.get("entrez_id"))
            if (9606, gid) not in by_id:
                continue
            for col in ("symbol", "prev_symbol", "alias_symbol", "name", "prev_name", "alias_name"):
                vals = _clean(row.get(col)).split("|")
                dest = descriptions if col.endswith("name") else aliases
                for val in vals:
                    _put(dest, 9606, val, gid)
    return aliases, descriptions


def _uniprot_url(taxid):
    # Reviewed entries limit noisy aliases and isoform ambiguity.
    params = {
        "query": f"(organism_id:{taxid}) AND (reviewed:true)",
        "format": "tsv",
        "fields": "accession,gene_primary,gene_synonym,protein_name",
        "compressed": "true",
    }
    return UNIPROT_ENDPOINT + "?" + urlencode(params)


def _load_uniprot(cache_dir, by_id, official, synonyms, descriptions):
    """Map UniProt alternative protein names via uniquely grounded gene names.

    Only protein names with a unique species-specific NCBI gene are eligible.
    Conflicting UniProt names are stored as multi-ID and rejected downstream.
    """
    mapping = defaultdict(set)
    for taxid in SPECIES_TAXIDS.values():
        path = _download(_uniprot_url(taxid), cache_dir / f"uniprot_reviewed_{taxid}.tsv.gz", min_bytes=200)
        with gzip.open(path, "rt", encoding="utf-8", errors="replace", newline="") as f:
            rows = csv.DictReader(f, delimiter="\t")
            for row in rows:
                primary = _clean(row.get("Gene Names (primary)"))
                alternatives = _clean(row.get("Gene Names (synonym)"))
                candidates = set()
                for name in [primary] + alternatives.split():
                    if not name:
                        continue
                    k = (taxid, _key(name))
                    for pool in (official, synonyms):
                        ids = pool.get(k, set())
                        if len(ids) == 1:
                            candidates.update(ids)
                            break
                if len(candidates) != 1:
                    continue
                gid = next(iter(candidates))
                protein = _clean(row.get("Protein names"))
                # UniProt names commonly format: Recommended Name: Full=...;
                # Alternative Name: Full=...; Short=...;
                names = re.findall(r"(?:Full|Short)=([^;\[\]]+)", protein)
                if not names and protein:
                    names = [protein.split(" (")[0]]
                for name in names:
                    name = re.sub(r"\s*\{[^}]*\}\s*$", "", name).strip()
                    if len(name) > 2:
                        _put(mapping, taxid, name, gid)
    return mapping


def _references(cache_dir, use_hgnc=True, use_uniprot=True):
    signature = (str(cache_dir.resolve()), use_hgnc, use_uniprot)
    if signature in _REFERENCE_CACHE:
        return _REFERENCE_CACHE[signature]
    by_id, official, synonyms, descriptions = _load_ncbi(cache_dir)
    # Keep per-reference lookup tables separate *before* merging aliases.
    # The existing normalization priorities and candidate pools are unchanged.
    ncbi_aliases = {k: set(v) for k, v in synonyms.items()}
    ncbi_descriptions = {k: set(v) for k, v in descriptions.items()}
    hgnc_aliases, hgnc_descriptions = defaultdict(set), defaultdict(set)
    extras = defaultdict(set)
    if use_hgnc:
        try:
            hg_alias, hg_desc = _load_hgnc(cache_dir, by_id)
            hgnc_aliases, hgnc_descriptions = hg_alias, hg_desc
            for k, v in hg_alias.items():
                synonyms[k].update(v)
            for k, v in hg_desc.items():
                descriptions[k].update(v)
        except Exception as exc:
            warnings.warn(f"HGNC unavailable; using NCBI only for these aliases: {exc}")
    if use_uniprot:
        try:
            extras = _load_uniprot(cache_dir, by_id, official, synonyms, descriptions)
        except Exception as exc:
            warnings.warn(f"UniProt unavailable; using NCBI/HGNC aliases: {exc}")
    provenance = {
        "ncbi_aliases": ncbi_aliases,
        "ncbi_descriptions": ncbi_descriptions,
        "hgnc_aliases": hgnc_aliases,
        "hgnc_descriptions": hgnc_descriptions,
    }
    _REFERENCE_CACHE[signature] = (by_id, official, synonyms, descriptions, extras, provenance)
    return _REFERENCE_CACHE[signature]


def _pmid_species(pmid_species, pmid):
    if pmid_species is None:
        return ""
    p = _id(pmid)
    val = pmid_species.get(p, pmid_species.get(int(p)) if p.isdigit() else None)
    if val is None:
        return ""
    vals = [val] if isinstance(val, str) else list(val)
    matches = {str(v).lower() for v in vals} & set(SPECIES_TAXIDS)
    return next(iter(matches)) if len(matches) == 1 else ("ambiguous" if matches else "")


# ---------------------------------------------------------------------------
# MirGeneDB 3.x: cached browse table + GFF loci, mapped to NCBI Gene
# ---------------------------------------------------------------------------
_MIR_CACHE = {}

def _is_mirna(v):
    s = _clean(v)
    return bool(re.match(r"^(?:(?:hsa|rno|mmu)[-_])?(?:mir(?:na|cro-?rna)?|let)[-_]?\d", s, re.I)) or bool(re.match(r"^MIRLET\d", s, re.I))

def _mir_key(v):
    return re.sub(r"[^a-z0-9]", "", _clean(v).casefold())

def _mir_names(raw, species):
    """Conservative precursor-name candidates; mature arm is checked separately."""
    s = re.sub(r"^(hsa|rno|mmu)[-_]", "", _clean(raw), flags=re.I).lower()
    arm_match = re.search(r"[-_](3p|5p)(?:\*)?$", s)
    arm = arm_match.group(1) if arm_match else ""
    if arm_match:
        s = s[:arm_match.start()]
    s = re.sub(r"^microrna[-_]?", "mir-", s)
    s = re.sub(r"^mirna[-_]?", "mir-", s)
    if s.startswith("mirlet"):
        s = "let-" + s[6:]
    if s.startswith("mir") and re.match(r"^mir\d", s):
        s = "mir-" + s[3:]
    return {_mir_key(species + '-' + s)}, arm

def _mir_locus_key(v):
    s = re.sub(r"_(?:pre|[35]p)\*?$", "", _clean(v), flags=re.I)
    s = re.sub(r"-v\d+$", "", s, flags=re.I)
    return _mir_key(s)

def _load_mirgenedb(cache_dir, species, by_id, official, synonyms):
    signature = (str(cache_dir.resolve()), species)
    if signature in _MIR_CACHE:
        return _MIR_CACHE[signature]
    browse_url = f"https://mirgenedb.org/browse/{species}"
    gff_url = f"https://mirgenedb.org/gff/{species}?all=1&sort=pos"
    browse_path = _download(browse_url, cache_dir / f"mirgenedb_{species}_browse.html", min_bytes=100)
    gff_path = _download(gff_url, cache_dir / f"mirgenedb_{species}.gff3", min_bytes=100)
    tables = pd.read_html(StringIO(browse_path.read_text(encoding="utf-8", errors="replace")))
    browse = None
    for table in tables:
        cols = [" ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in table.columns]
        mir_cols = [i for i, c in enumerate(cols) if "mirgenedb" in c.lower() and "id" in c.lower()]
        base_cols = [i for i, c in enumerate(cols) if "mirbase" in c.lower() and "id" in c.lower()]
        if mir_cols and base_cols:
            browse = table.iloc[:, [mir_cols[0], base_cols[0]]].dropna()
            break
    if browse is None:
        raise ValueError(f"MirGeneDB browse columns not found for {species}")

    loci = defaultdict(lambda: {"pre": set(), "arms": set()})
    for line in gff_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) < 9 or parts[2] not in {"pre_miRNA", "miRNA"}:
            continue
        attrs = dict(v.split("=", 1) for v in parts[8].split(";") if "=" in v)
        fid = attrs.get("ID", "")
        if not fid:
            continue
        locus = loci[_mir_locus_key(fid)]
        if parts[2] == "pre_miRNA":
            locus["pre"].add(fid)
        else:
            m = re.search(r"_([35]p)\*?$", fid, re.I)
            if m:
                locus["arms"].add(m.group(1).lower())

    tid = SPECIES_TAXIDS[species]
    index = defaultdict(dict)
    # Build a canonical NCBI lookup strictly from symbols/synonyms, no fuzzy mapping.
    for mirgene, precursor in browse.itertuples(index=False, name=None):
        mirgene, precursor = _clean(mirgene), _clean(precursor)
        locus = loci.get(_mir_locus_key(mirgene))
        if not locus or not precursor:
            continue
        base_key, _ = _mir_names(precursor, species)
        keys = set(base_key)
        # mirbase precursor usually already contains prefix and may have an arm.
        keys.add(_mir_key(precursor))
        local = re.sub(rf"^{species}[-_]", "", precursor, flags=re.I)
        ncbi_keys = {_key(local)}
        if local.lower().startswith('let-'):
            ncbi_keys.add(_key('mir' + local))
        if local.lower().startswith('mir-'):
            ncbi_keys.add(_key('mir' + local[4:]))
        ids = set()
        for k in ncbi_keys:
            ids.update(official.get((tid, k), set()))
            ids.update(synonyms.get((tid, k), set()))
        ids = {v for v in ids if (tid, v) in by_id}
        for key in keys:
            index[key][mirgene] = (set(ids), set(locus['arms']))
    _MIR_CACHE[signature] = index
    return index

def _mir_resolve(raw, species, mir_index):
    keys, arm = _mir_names(raw, species)
    matched = {}
    for key in keys:
        matched.update(mir_index.get(key, {}))
    if arm:
        matched = {locus: val for locus, val in matched.items() if arm in val[1]}
    if not matched:
        return set(), 'mirgenedb_unresolved'
    if len(matched) != 1:
        return set(), 'mirgenedb_ambiguous_locus'
    ids = next(iter(matched.values()))[0]
    if len(ids) != 1:
        return set(), 'mirgenedb_no_unique_ncbi_gene'
    return ids, 'mirgenedb_unique'


def normalize_gene_symbols(df, cache_dir=CACHE_DIR, pmid_species=None,
                           default_species="hsa", strict_conflicts=True,
                           use_hgnc=True, use_uniprot=True, use_mirgenedb=True):
    """Return df with unique, species-aware official gene symbols.

    Priorities: prior verified miRNA > MirGeneDB precursor locus + NCBI Gene
    > official NCBI symbol > NCBI/HGNC alias
    > NCBI/HGNC gene description > UniProt protein name.
    Each mapping must be unique and species-consistent.
    Gene IDs are used for validation, not to choose the target when ambiguous.
    Does not write audit files.
    """
    df = df.copy()
    if "geneSymbol" not in df.columns:
        return df
    for col in ("geneID", "geneName"):
        if col not in df.columns:
            df[col] = ""
    for col in ("geneSymbol", "geneID", "geneName"):
        if col + "_original" not in df.columns:
            df[col + "_original"] = df[col]
    if "normalization_status" not in df.columns:
        df["normalization_status"] = "unchanged"

    # PMID-aware inference from explicit species-prefixed miRNAs in this DataFrame.
    # Caller-supplied pmid_species takes precedence. Mixed species remain ambiguous.
    if pmid_species is None and "pmid" in df.columns:
        inferred_species = defaultdict(set)
        for pmid_value, symbol_value in df[["pmid", "geneSymbol"]].itertuples(index=False, name=None):
            if _is_mirna(symbol_value):
                explicit = _species_from_text(symbol_value)
                if explicit:
                    inferred_species[_id(pmid_value)].add(explicit)
        pmid_species = inferred_species

    by_id, official, aliases, descriptions, proteins, provenance = _references(
        Path(cache_dir), use_hgnc, use_uniprot)
    out = {c: df[c].tolist() for c in ("geneSymbol", "geneID", "geneName", "normalization_status")}
    statuses, verified, species_list = [], [], []
    resolution_db_trace = [""] * len(df)
    mir_indexes = {}
    # Lazy initialization: only download MirGeneDB for species encountered.
    # A failed optional download retains other references but never claims a MirGeneDB match.
    mir_failures = set()

    for i, (_, row) in enumerate(df.iterrows()):
        symbol = _clean(row.get("geneSymbol"))
        name = _clean(row.get("geneName"))
        gid = _id(row.get("geneID"))
        status, is_verified, species = "unchanged", False, ""
        if not symbol and not name:
            statuses.append(status); verified.append(False); species_list.append("")
            continue

        mir_status = _clean(row.get("mirna_mapping_status")).lower()
        previous_status = _clean(row.get("normalization_status")).lower()
        valid_mir = (previous_status == "high_confidence_normalized" and gid and
                     (mir_status in {"high_confidence", "applied", "mapped", "verified",
                                     "high_confidence_normalized", "unique", "mirgenedb_unique"}
                      or (_is_mirna(row.get("geneSymbol_original")) and mir_status not in {
                          "", "unresolved", "ambiguous", "secondary_unique_candidate",
                          "not_mirna_or_not_identified", "not_found"})))
        if valid_mir:
            statuses.append("existing_mirna_normalization")
            verified.append(True); species_list.append(_species_from_text(symbol))
            continue

        exp_sym = _species_from_text(symbol)
        exp_name = _species_from_text(name)
        if exp_sym and exp_name and exp_sym != exp_name:
            statuses.append("species_conflict_review"); verified.append(False); species_list.append("")
            continue
        species = exp_sym or exp_name
        pmid_sp = _pmid_species(pmid_species, row.get("pmid"))
        if pmid_sp == "ambiguous" and not species:
            statuses.append("ambiguous_species"); verified.append(False); species_list.append("")
            continue
        if species and pmid_sp not in {"", "ambiguous", species}:
            statuses.append("species_conflict_review"); verified.append(False); species_list.append(species)
            continue
        if not species:
            # Existing Gene ID can independently establish the organism; never force human otherwise.
            found = {tid for tid in SPECIES_TAXIDS.values() if (tid, gid) in by_id} if gid else set()
            if len(found) == 1:
                species = TAXID_SPECIES[next(iter(found))]
            elif len(found) > 1 and not pmid_sp:
                statuses.append("ambiguous_species"); verified.append(False); species_list.append("")
                continue
            else:
                species = pmid_sp or default_species
        if species not in SPECIES_TAXIDS:
            statuses.append("unsupported_species"); verified.append(False); species_list.append(species)
            continue
        tid = SPECIES_TAXIDS[species]
        local_symbol = re.sub(r"^(hsa|rno|mmu)[-_]", "", symbol, flags=re.I).strip()
        ksym = _key(local_symbol)
        is_mir = _is_mirna(symbol) or _is_mirna(name)
        if is_mir and use_mirgenedb:
            if species not in mir_indexes and species not in mir_failures:
                try:
                    mir_indexes[species] = _load_mirgenedb(
                        Path(cache_dir), species, by_id, official, aliases)
                except Exception as exc:
                    warnings.warn(f"MirGeneDB unavailable for {species}: {exc}")
                    mir_failures.add(species)
            if species in mir_indexes:
                mir_candidates, mir_state = _mir_resolve(symbol or name, species, mir_indexes[species])
                # A known ambiguous locus is not disambiguated by unrelated protein aliases.
                if mir_state == "mirgenedb_ambiguous_locus":
                    statuses.append(mir_state); verified.append(False); species_list.append(species)
                    continue
                if len(mir_candidates) == 1:
                    target = next(iter(mir_candidates))
                    if gid and gid != target and strict_conflicts:
                        statuses.append("gene_id_conflict_review"); verified.append(False); species_list.append(species)
                        continue
                    proposed, desc = by_id[(tid, target)]
                    if proposed != symbol:
                        out["geneSymbol"][i] = proposed
                    if not gid:
                        out["geneID"][i] = target
                    if not name or _key(name) == _key(symbol):
                        out["geneName"][i] = desc
                    if proposed != symbol or not gid:
                        out["normalization_status"][i] = "high_confidence_normalized"
                    statuses.append("mapped_mirgenedb" if proposed != symbol else "already_official_mirna")
                    verified.append(True); species_list.append(species)
                    continue
                # Fall through to exact NCBI aliases when no unique MirGeneDB locus.
        # Do not merge candidates from independent name fields. A resolved symbol wins;
        # the name is a fallback and only a separately resolved contradiction triggers review.
        candidates, source = set(), ""
        resolution_key = ksym
        for src, mapping in (("official", official), ("synonym", aliases),
                             ("gene_description", descriptions), ("uniprot_protein", proteins)):
            found = mapping.get((tid, ksym), set())
            if found:
                candidates, source = set(found), src
                break
        if not candidates and name:
            kname = _key(name)
            for src, mapping in (("name_official", official), ("name_synonym", aliases),
                                 ("name_description", descriptions), ("name_uniprot", proteins)):
                found = mapping.get((tid, kname), set())
                if found:
                    candidates, source = set(found), src
                    resolution_key = kname
                    break
        if len(candidates) != 1:
            status = "ambiguous_reference_mapping" if candidates else "unresolved"
            statuses.append(status); verified.append(False); species_list.append(species)
            continue
        target = next(iter(candidates))
        if gid and gid != target and strict_conflicts:
            statuses.append("gene_id_conflict_review"); verified.append(False); species_list.append(species)
            continue
        if name and _key(name) != ksym:
            # Name contradictions are checked, but absence from reference is not a veto.
            kname = _key(name)
            name_ids = set()
            for mapping in (official, aliases, descriptions):
                name_ids = mapping.get((tid, kname), set())
                if name_ids:
                    break
            if name_ids and target not in name_ids and source != "official":
                statuses.append("symbol_name_conflict_review"); verified.append(False); species_list.append(species)
                continue
        # Resolve precisely which reference(s) support the *selected* alias or name.
        if source in {"synonym", "name_synonym", "gene_description", "name_description"}:
            key_prefix = "aliases" if "synonym" in source else "descriptions"
            ncbi_pool = provenance[f"ncbi_{key_prefix}"]
            hgnc_pool = provenance[f"hgnc_{key_prefix}"]
            in_ncbi = target in ncbi_pool.get((tid, resolution_key), set())
            in_hgnc = target in hgnc_pool.get((tid, resolution_key), set())
            if in_ncbi and in_hgnc:
                resolution_db_trace[i] = "NCBI Gene + HGNC (both support alias)"
            elif in_hgnc:
                resolution_db_trace[i] = "HGNC"
            elif in_ncbi:
                resolution_db_trace[i] = "NCBI Gene"
        proposed, desc = by_id[(tid, target)]
        is_verified = True
        symbol_changed = symbol != proposed
        if symbol_changed:
            out["geneSymbol"][i] = proposed
        if not gid:
            out["geneID"][i] = target
        if not name or _key(name) == _key(symbol):
            out["geneName"][i] = desc
        if symbol_changed or not gid:
            out["normalization_status"][i] = "high_confidence_normalized"
        status = f"mapped_{source}" if symbol_changed else "already_official_symbol"
        statuses.append(status); verified.append(is_verified); species_list.append(species)

    for c, values in out.items():
        df[c] = values
    df["gene_symbol_resolution"] = statuses
    df["gene_identity_verified"] = verified
    df["gene_symbol_species"] = species_list
    # Provenance of the accepted *second-stage* match; unchanged validations
    # are included and should not be confused with fields that were changed.
    def _db_from_status(st):
        if st in {"mapped_mirgenedb", "already_official_mirna"}:
            return "MirGeneDB + NCBI Gene"
        if st == "existing_mirna_normalization":
            return "Previously verified miRNA (source not recorded here)"
        if st in {"mapped_official", "mapped_name_official", "already_official_symbol"}:
            return "NCBI Gene"
        if "uniprot" in st:
            return "UniProt (NCBI Gene anchored)"
        if st.startswith(("mapped_",)):
            return "Unknown merged alias provenance"
        return "No accepted mapping (or unresolved)"
    df["gene_resolution_database"] = [
        trace if trace else _db_from_status(status)
        for trace, status in zip(resolution_db_trace, statuses)
    ]
    if "gene_key" in df.columns:
        df = df.drop(columns=["gene_key"])
    print("Gene symbols: changed=", sum(a != b for a, b in zip(df["geneSymbol"], df["geneSymbol_original"])),
          "verified=", sum(verified), "ambiguous/conflicting=",
          sum(s.startswith(("ambiguous", "gene_id_conflict", "symbol_name_conflict")) for s in statuses))
    return df


In [ ]:
# Copied from Bedrock_preAnalysis section 8; helper logic unchanged.
# Intended order within the combined analysis:
#   update_dataframe -> normalize_gene_symbols (external DBs)
#   -> update_gene_symbol_ (custom CTD fallback) -> dedup -> optional filter
import re
from typing import Optional
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Small pure helpers
# ---------------------------------------------------------------------------

def _symbol_numeric_suffix(symbol) -> Optional[str]:
    """
    Extract the portion of a GeneSymbol starting from its first digit.
        FKBP1A -> "1A"
        IL6    -> "6"
        TP53   -> "53"
    Returns None if no digit is found or input is NaN.
    """
    if pd.isna(symbol):
        return None
    m = re.search(r"\d.*", str(symbol))
    return m.group(0) if m else None


def _tokenize(name) -> list:
    """Split a (already-cleaned) name string into lowercase tokens."""
    if pd.isna(name) or not str(name).strip():
        return []
    return str(name).strip().lower().split()


def _remove_parentheses(name) -> str:
    """
    Strip parenthesised substrings (including the parentheses themselves).
        "glyceraldehyde-3-phosphate dehydrogenase (GAPDH)"
        -> "glyceraldehyde-3-phosphate dehydrogenase"
    """
    if pd.isna(name):
        return ""
    return re.sub(r"\s*\(.*?\)", "", str(name)).strip()


def _suffix_in_tokens(suffix: Optional[str], tokens: list) -> bool:
    """
    Return True when *suffix* (e.g. "53") is represented in *tokens*.

    Single-token names  : suffix must appear *anywhere* in the token.
    Multi-token names   : at least one token must *end with* suffix.
    """
    if not suffix:
        return False
    suffix = suffix.lower()
    if len(tokens) == 1:
        return suffix in tokens[0]
    return any(tok.endswith(suffix) for tok in tokens)


def _normalize(s) -> str:
    """Lowercase, strip, coerce NaN -> empty string."""
    if pd.isna(s):
        return ""
    return str(s).strip().lower()


def _alt_count(alt_ids_str) -> int:
    """Count non-empty pipe-separated alternative gene IDs."""
    return len([a for a in str(alt_ids_str).split("|") if a.strip()])


def _symbol_is_missing(symbol) -> bool:
    """Return True if a geneSymbol value should be treated as absent."""
    if pd.isna(symbol):
        return True
    return str(symbol).strip() in ("", "-")


def _strip_dot_suffix(symbol) -> Optional[str]:
    """
    If a symbol contains a dot, return the portion before it.
        BRCA1.2  -> "BRCA1"
        TP53.1   -> "TP53"
        BRCA1    -> None  (no dot present)
    """
    if pd.isna(symbol):
        return None
    s = str(symbol).strip()
    if "." in s:
        return s.split(".")[0]
    return None


# ---------------------------------------------------------------------------
# CTD pre-processing  (call once, reuse across batches)
# ---------------------------------------------------------------------------

def prepare_ctd(ctd_genes: pd.DataFrame) -> dict:
    """
    Build fast lookup structures from the CTD gene table.

    Returns a dict with:
        "df"           : cleaned CTD dataframe with helper columns
        "id_index"     : {gene_id_str          -> list[row_index]}
        "symbol_index" : {normalised_symbol    -> list[row_index]}
        "name_index"   : {normalised_gene_name -> list[row_index]}
        "synonym_index": {normalised_synonym   -> list[row_index]}
    """
    ctd = ctd_genes.copy()
    ctd["GeneID"]       = ctd["GeneID"].astype(str)
    ctd["_symbol_norm"] = ctd["# GeneSymbol"].apply(_normalize)
    ctd["_name_norm"]   = ctd["GeneName"].apply(_normalize)
    ctd["_alt_count"]   = ctd["AltGeneIDs"].fillna("").apply(_alt_count)
    ctd["_syns"]        = ctd["Synonyms"].fillna("").str.lower().str.split("|")

    id_index: dict = {}
    for i, gid in ctd["GeneID"].items():
        if gid:
            id_index.setdefault(gid, []).append(i)

    symbol_index: dict = {}
    for i, sym in ctd["_symbol_norm"].items():
        if sym:
            symbol_index.setdefault(sym, []).append(i)

    name_index: dict = {}
    for i, name in ctd["_name_norm"].items():
        if name:
            name_index.setdefault(name, []).append(i)

    synonym_index: dict = {}
    for i, syns in ctd["_syns"].items():
        for s in syns:
            s = s.strip()
            if s:
                synonym_index.setdefault(s, []).append(i)

    return {
        "df": ctd,
        "id_index": id_index,
        "symbol_index": symbol_index,
        "name_index": name_index,
        "synonym_index": synonym_index,
    }


# ---------------------------------------------------------------------------
# Per-row resolution helpers
# ---------------------------------------------------------------------------

def _resolve_by_id(gene_id: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 1: exact GeneID match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["id_index"].get(gene_id, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_symbol(symbol_norm: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 2: exact normalised GeneSymbol match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["symbol_index"].get(symbol_norm, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_name(name_norm: str, name_tokens: list, ctd_lookup: dict) -> Optional[pd.Series]:
    """
    Priority 3: match via GeneName or Synonyms with alt_count tiebreaking.

    Cases
    -----
    1. Only GeneName hits  : prefer suffix-matching candidates; tiebreak by alt_count
    2. Both hits           : compare total alt_count of each group; pick winner's best
    3. Only Synonym hits   : pick highest alt_count
    """
    if not name_norm:
        return None

    ctd_df        = ctd_lookup["df"]
    name_index    = ctd_lookup["name_index"]
    synonym_index = ctd_lookup["synonym_index"]

    gn_rows  = ctd_df.loc[name_index.get(name_norm, [])]
    syn_rows = ctd_df.loc[synonym_index.get(name_norm, [])]

    # Deduplicate: remove synonym hits already present in gene-name hits
    if not gn_rows.empty and not syn_rows.empty:
        syn_rows = syn_rows[~syn_rows.index.isin(gn_rows.index)]

    # ---- Case 1: only GeneName hits ----
    if not gn_rows.empty and syn_rows.empty:
        suffix_matches = [
            r for _, r in gn_rows.iterrows()
            if _suffix_in_tokens(_symbol_numeric_suffix(r["# GeneSymbol"]), name_tokens)
        ]
        best_df = pd.DataFrame(suffix_matches) if suffix_matches else gn_rows
        return best_df.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 2: both hits ----
    if not gn_rows.empty and not syn_rows.empty:
        if gn_rows["_alt_count"].sum() >= syn_rows["_alt_count"].sum():
            return gn_rows.sort_values("_alt_count", ascending=False).iloc[0]
        else:
            return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 3: only Synonym hits ----
    if not syn_rows.empty:
        return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    return None


# ---------------------------------------------------------------------------
# Dot-symbol resolution helper
# ---------------------------------------------------------------------------

def _apply_dot_check(best: pd.Series, ctd_lookup: dict) -> pd.Series:
    """
    If the resolved symbol contains a dot (e.g. BRCA1.2), strip the suffix
    and re-look up the base symbol in CTD to get the correct GeneID/GeneName.
    Returns the corrected row, or the original row with the dot stripped if
    no CTD entry is found for the base.
    """
    base = _strip_dot_suffix(best["# GeneSymbol"])
    if base is None:
        return best  # no dot, nothing to do

    hit = _resolve_by_symbol(_normalize(base), ctd_lookup)
    if hit is not None:
        return hit

    # No CTD entry for base symbol — strip the dot suffix in-place
    best = best.copy()
    best["# GeneSymbol"] = base
    return best


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------
def update_gene_symbol_(
    df: pd.DataFrame,
    ctd_genes: pd.DataFrame,
    ctd_lookup: Optional[dict] = None,
) -> pd.DataFrame:

    df = df.copy()

    df["oldGeneID"]     = df.get("geneID")
    df["oldGeneName"]   = df.get("geneName")
    df["oldGeneSymbol"] = df.get("geneSymbol")

    # ✅ Treat "-", " -", "- " etc. as missing across key gene columns
    for col in ["geneID", "geneSymbol", "geneName"]:
        if col in df.columns:
            df[col] = df[col].replace(r"^\s*-\s*$", "", regex=True).fillna("")

    if ctd_lookup is None:
        ctd_lookup = prepare_ctd(ctd_genes)

    # Vectorised pre-normalisation
    df["_name_norm"] = (
        df.get("geneName", pd.Series(dtype=str))
        .apply(_remove_parentheses)
        .apply(_normalize)
    )
    df["_symbol_norm"] = (
        df.get("geneSymbol", pd.Series(dtype=str))
        .apply(_normalize)
    )

    flags       = np.full(len(df), 2, dtype=int)
    new_symbols = df["geneSymbol"].copy() if "geneSymbol" in df.columns else pd.Series(index=df.index, dtype=str)
    new_ids     = df["geneID"].astype(str).copy()
    new_names   = df["geneName"].copy()   if "geneName"   in df.columns else pd.Series(index=df.index, dtype=str)

    for pos, (idx, row) in enumerate(df.iterrows()):
        raw_symbol  = row.get("geneSymbol")
        gene_id     = str(row.get("geneID", "")).strip()
        symbol_norm = row["_symbol_norm"]
        name_norm   = row["_name_norm"]
        name_tokens = _tokenize(name_norm)

        # Already valid symbol
        if not _symbol_is_missing(raw_symbol):
            if not gene_id:  # symbol present but ID missing — try to resolve ID from symbol
                best = _resolve_by_symbol(symbol_norm, ctd_lookup)
                if best is not None:
                    best = _apply_dot_check(best, ctd_lookup)
                    new_symbols.iloc[pos] = best["# GeneSymbol"]
                    new_ids.iloc[pos]     = best["GeneID"]
                    new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue
    
        # Priority 1: GeneID
        if gene_id:
            best = _resolve_by_id(gene_id, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 2: GeneSymbol
        if symbol_norm:
            best = _resolve_by_symbol(symbol_norm, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 3a: treat geneName as a symbol
        best = _resolve_by_symbol(name_norm, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue

        # Priority 3b: GeneName / Synonym
        best = _resolve_by_name(name_norm, name_tokens, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0

    df["geneSymbol"]     = new_symbols.values
    df["geneID"]         = new_ids.values
    df["geneName"]       = new_names.values
    df["geneSymbolFlag"] = flags

    df.drop(columns=["_name_norm", "_symbol_norm"], inplace=True)
    return df


In [ ]:
data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover'
ctd_data_path = data_path + '/CTD_data'
ctd_genes = pd.read_csv(f"{ctd_data_path}/CTD_genes.csv", skiprows=27)
# Build the CTD symbol/name/synonym indexes once for all 18 configurations.
CTD_SYMBOL_LOOKUP = prepare_ctd(ctd_genes)
print("CTD gene-symbol fallback lookup ready.")


## 3. CTD reference

In [ ]:
REFERENCE_PMIDS = set([
    '9118903', '15154616', '15811834', '15953866', '16393664',
    '17118201', '17183729', '17524151', '18280460', '18299717',
    '19114014', '19617202', '19654922', '20801540', '21294877',
    '22110480', '22112310', '22228119', '22562770', '22571318',
    '22747577', '22808131', '23006841', '23349674', '23568217',
    '24039778', '24403256', '24413757', '25361045', '25859307',
    '26039251', '26492236', '26714306', '27153807', '27397436',
    '27517156', '27736846', '28637469', '29045497', '29380240',
    '30104959', '30114287', '31064381', '31378678', '32298299',
    '34908893', '36638633', '37047793', '25100957', '19889203',
])

print("Reference PMID count:", len(REFERENCE_PMIDS))

In [ ]:
def read_ctd_csv(filename):
    path = CTD_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing CTD reference: {path}")
    return pd.read_csv(path, skiprows=27, low_memory=False).drop(index=0, errors="ignore").copy()

ctd_chemicals = read_ctd_csv("CTD_chemicals.csv")
ctd_genes = read_ctd_csv("CTD_genes.csv")
ctd_raw = read_ctd_csv("CTD_chem_gene_ixns.csv")

# Original CTD reference-generation semantics
ctd_raw["GeneID"] = pd.to_numeric(ctd_raw["GeneID"], errors="coerce").astype("Int64").astype("string")
ctd_raw["ChemicalID"] = ctd_raw["ChemicalID"].astype("string")
ctd_raw["InteractionActionsList"] = ctd_raw["InteractionActions"].fillna("").apply(
    lambda s: [x.replace("^", " ") for x in str(s).split("|") if x.strip()]
)
ctd_raw["pmidsList"] = ctd_raw["PubMedIDs"].fillna("").apply(
    lambda s: [x.strip() for x in str(s).split("|") if x.strip()]
)
ctd_raw["ref_pmids"] = ctd_raw["pmidsList"].apply(lambda ids: [x for x in ids if x in REFERENCE_PMIDS])
ctd_ixns = (ctd_raw[ctd_raw["ref_pmids"].apply(len) > 0]
    .explode("ref_pmids").rename(columns={"ref_pmids": "pmid"}))
ctd_ixns = ctd_ixns[ctd_ixns["InteractionActionsList"].apply(len) == 1].copy()
ctd_ixns = ctd_ixns.dropna(subset=["ChemicalID", "GeneID"])
ctd_ixns = ctd_ixns.drop_duplicates(
    subset=["pmid", "ChemicalID", "GeneID", "InteractionActions"]
).reset_index(drop=True)

print("CTD reference rows:", len(ctd_ixns))
print("Reference PMIDs:", ctd_ixns["pmid"].nunique())
ctd_ixns.to_csv(SAVE_DIR / "ctd_reference.csv", index=False)

## 4. Deduplication — exact original implementation

In [ ]:
def drop_duplicates_(df): # drop duplicates globally with normalized relationship
    pmids = [
    9118903, 15154616, 15811834, 15953866, 16393664,
    17118201, 17183729, 17524151, 18280460, 18299717,
    19114014, 19617202, 19654922, 20801540, 21294877,
    22110480, 22112310, 22228119, 22562770, 22571318,
    22747577, 22808131, 23006841, 23349674, 23568217,
    24039778, 24403256, 24413757, 25361045, 25859307,
    26039251, 26492236, 26714306, 27153807, 27397436,
    27517156, 27736846, 28637469, 29045497, 29380240,
    30104959, 30114287, 31064381, 31378678, 32298299,
    34908893, 36638633, 37047793, 25100957, 19889203,
    ]    
    df = df.copy()
    df['pmid'] = pd.to_numeric(df['pmid'], errors='coerce').astype('Int64')
    df = df[df['pmid'].isin(pmids)].copy()
    
    # Normalize relationship (same as in match function)
    def normalize_relationship(s):
        if pd.isna(s):
            return s
        s = str(s).lower().strip()
        s = s.replace("^", " ")
        s = s.replace("increases", "increase")
        s = s.replace("decreases", "decrease")
        s = s.replace("affects", "affect")
        s = " ".join(s.split())
        return s
    
    df['relationship_norm'] = df['relationship'].apply(normalize_relationship)
    
    # Build gene_key (same as in match function)
    df["geneSymbol_clean"] = (
        df["geneSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\..+$", "", regex=True)
    )
    
    def make_gene_key(row):
        sym = row["geneSymbol_clean"]
        if pd.notna(sym) and sym not in ("", "NAN", "NA"):
            return sym
        if pd.notna(row.get("geneID")):
            try:
                return str(int(row["geneID"]))
            except:
                return pd.NA
        return pd.NA
    
    df["gene_key"] = df.apply(make_gene_key, axis=1)
    
    # Section priority
    section_priority = {
        "TABLE": 0,
        "FIG_TABLE": 1,
        "FIG": 2,
        "SUPPLEMENT": 3,
        "TITLE": 4,
        "ABSTRACT": 5,
        "RESULTS": 6,
        "DISCUSS": 7,
        "METHODS": 8,
        "INTRO": 8,
        "CONCL": 8
    }
    
    df['_section_priority'] = df['section_type'].map(section_priority).fillna(100)
    df = df.reset_index(drop=True)
    df['_row_order'] = df.index
    
    # Sort by priority, then original row order
    df = df.sort_values(['_section_priority', '_row_order'], ascending=[True, True])
    
    # Drop duplicates using NORMALIZED MERGE KEYS
    df = df.drop_duplicates(
        subset=['pmid', 'gene_key', 'chemicalID', 'relationship_norm'], 
        keep='first'
    )

    df = df.drop_duplicates(
        subset=['pmid', 'gene_key', 'chemicalID', 'relationship_norm'], 
    )
    
    # KEEP relationship_norm and gene_key for downstream use
    # ONLY drop temporary columns
    df = df.drop(columns=['_section_priority', '_row_order', 'geneSymbol_clean'])
    
    return df

## 5. Quality filtering

In [ ]:
def update_dataframe(sample):
    """
    Normalize DataFrame columns without removing rows.
    """

    sample = sample.copy()

    # Normalize text columns
    string_cols = [
        "source",
        "text",
        "relationship",
        "chemicalName",
        "chemicalID",
        "geneSymbol",
        "geneID",
        "section_type",
        "chemical_map",
        "gene_map",
        "reasoning",
    ]

    for col in string_cols:
        if col not in sample.columns:
            sample[col] = ""

        sample[col] = (
            sample[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

    # Normalize PMID
    sample["pmid"] = (
        sample["pmid"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    # Normalize section type
    sample["section_type"] = (
        sample["section_type"]
        .str.upper()
        .str.strip()
    )

    # Normalize chemical IDs
    sample["chemicalID"] = (
        sample["chemicalID"]
        .str.upper()
        .replace({
            "D000072317": "D013749"
        })
    )

    # CASE-INSENSITIVE GENE SYMBOL MATCHING
    # Preserve geneSymbol_original if it exists.
    sample["geneSymbol"] = (
        sample["geneSymbol"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # Preserve original relationship
    sample["relationship_original"] = sample["relationship"]

    # Normalize relationship formatting
    sample["relationship"] = (
        sample["relationship"]
        .str.lower()
        .str.replace("^", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Sort if original index exists
    if "Unnamed: 0" in sample.columns:
        sample = sample.sort_values("Unnamed: 0")

    return sample.reset_index(drop=True)

def id_in_map(identifier, mapping):
    """
    Check whether an identifier appears as a complete token
    inside an entity map.

    Prevents gene ID 123 from matching 1234.
    """

    identifier = str(identifier).strip()

    if identifier.lower() in {"", "nan", "none", "<na>"}:
        return False

    if pd.isna(mapping):
        return False

    mapping = str(mapping)

    pattern = (
        r"(?<![A-Za-z0-9])"
        + re.escape(identifier)
        + r"(?![A-Za-z0-9])"
    )

    return bool(
        re.search(
            pattern,
            mapping,
            flags=re.IGNORECASE
        )
    )

def normalize_relationship(relationship):
    """
    Normalize CTD-style relationship prefixes.

    Examples:
        increases^expression -> increase expression
        decreases expression -> decrease expression
        affects^activity     -> affect activity
    """

    relationship = str(relationship).lower().strip()
    relationship = relationship.replace("^", " ")
    relationship = re.sub(r"\s+", " ", relationship)

    relationship = re.sub(
        r"^increases\b", "increase", relationship
    )

    relationship = re.sub(
        r"^decreases\b", "decrease", relationship
    )

    relationship = re.sub(
        r"^affects\b", "affect", relationship
    )

    return relationship.strip()

def filter_relationships(group):
    """
    Remove generic affect relationships only when a
    directional relationship exists for the same:

        PMID + chemical + gene + action

    Preserve unrelated relationship actions.
    """

    group = group.copy()

    group["_relationship_norm"] = (
        group["relationship"]
        .apply(normalize_relationship)
    )

    parsed = group["_relationship_norm"].str.extract(
        r"^(increase|decrease|affect)\s+(.+)$"
    )

    group["_prefix"] = parsed[0]
    group["_action"] = parsed[1]

    # Actions with a specific directional relationship
    specific_actions = set(
        group.loc[
            group["_prefix"].isin([
                "increase",
                "decrease"
            ]),
            "_action"
        ].dropna()
    )

    # Drop generic only when same action has
    # a more specific directional prediction.
    drop_generic = (
        group["_prefix"].eq("affect")
        &
        group["_action"].isin(specific_actions)
    )

    return group.loc[
        ~drop_generic
    ].drop(
        columns=[
            "_relationship_norm",
            "_prefix",
            "_action"
        ]
    )

import re
import pandas as pd

def flag_caption_evidence(sample):
    sample = sample.copy()

    section = (
        sample["section_type"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
    )

    captions = (
        sample["caption"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    is_visual = section.isin(["FIG", "TABLE", "FIG_TABLE"])

    # Captions describing associations rather than
    # demonstrated treatment effects.
    association_pattern = (
        r"\b("
        r"association|associated with|correlation|correlated|"
        r"polymorphism|genotype|snp|"
        r"risk factor|susceptibility"
        r")\b"
    )

    # Captions primarily describing pathways,
    # functions, or gene lists.
    descriptive_pattern = (
        r"\b("
        r"gene ontology|pathway analysis|"
        r"functional enrichment|venn diagram|"
        r"network analysis|literature comparison|"
        r"previous studies"
        r")\b"
    )

    # Positive evidence cues, not sufficient proof.
    treatment_pattern = (
        r"\b("
        r"treated|treatment|exposed|exposure|"
        r"administered|incubated|dose|"
        r"control group|vehicle control"
        r")\b"
    )

    measurement_pattern = (
        r"\b("
        r"expression|abundance|activity|"
        r"fold.change|upregulated|downregulated|"
        r"increased|decreased|reduced|"
        r"western blot|rt.pcr|qpcr|"
        r"rna.seq|protein level"
        r")\b"
    )

    sample["caption_association_cue"] = (
        is_visual &
        captions.str.contains(
            association_pattern, regex=True
        )
    )

    sample["caption_descriptive_cue"] = (
        is_visual &
        captions.str.contains(
            descriptive_pattern, regex=True
        )
    )

    sample["caption_treatment_cue"] = (
        is_visual &
        captions.str.contains(
            treatment_pattern, regex=True
        )
    )

    sample["caption_measurement_cue"] = (
        is_visual &
        captions.str.contains(
            measurement_pattern, regex=True
        )
    )

    sample["caption_review_required"] = (
        is_visual &
        (
            sample["caption_association_cue"] |
            sample["caption_descriptive_cue"] |
            ~sample["caption_treatment_cue"]
        )
    )

    return sample
    
def filter_dataframe(sample, CTDchemicals, CTDgenes):
    """
    Filter low-quality chemical-gene relationships.

    Features:
        1. Case-insensitive gene symbol matching
        2. PMID-aware relationship grouping
        3. Action-specific generic relationship filtering
        4. CTD vocabulary validation
        5. Retain successfully normalized miRNAs
        6. Targeted negative-result filtering
        7. Exact entity-map identifier validation
        8. Less aggressive reasoning filtering

    Returns:
        Filtered DataFrame.
    """

    sample = sample.copy()

    # --------------------------------------------------------
    # Ensure case-insensitive gene matching even when
    # filter_dataframe() is called directly.
    # --------------------------------------------------------

    sample["geneSymbol"] = (
        sample["geneSymbol"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    print(f"Initial length of data: {len(sample)}")

    # ========================================================
    # 1. FILTER GENERIC RELATIONSHIPS BY SPECIFICITY
    # ========================================================

    # Remove chemicals with missing or invalid MeSH IDs
    sample["chemicalID"] = (
        sample["chemicalID"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )
    
    valid_chemical_id = sample["chemicalID"].str.fullmatch(
        r"[CD]\d+",
        na=False
    )
    
    sample = sample.loc[valid_chemical_id].copy()

    group_cols = [
        "pmid",
        "chemicalID",
        "geneSymbol"
    ]

    sample = (
        pd.concat(
            [
                filter_relationships(group)
                for _, group in sample.groupby(
                    group_cols,
                    dropna=False,
                    sort=False
                )
            ],
            ignore_index=True
        )
        if not sample.empty
        else sample.copy()
    )

    print(
        "After relationship specificity filtering:",
        len(sample)
    )

    # ========================================================
    # 4. DROP ROWS WITH INVALID ENTITY FLAGS
    # ========================================================

    flag_cols = [
        "chemicalFlag",
        "relationshipWarningFlag",
        "prefixMissingFlag",
        "geneSymbolFlag"
    ]

    for col in flag_cols:
        if col not in sample.columns:
            raise KeyError(
                f"Required flag column missing: {col}"
            )

    # Chemical, relationship, and prefix flags
    # must always be zero.
    flag_mask = pd.Series(
        True,
        index=sample.index
    )

    for col in [
        "chemicalFlag",
        "relationshipWarningFlag",
        "prefixMissingFlag"
    ]:
        flag_mask &= pd.to_numeric(
            sample[col],
            errors="coerce"
        ).eq(0)
        
    # ========================================================
    # 2. FILTER USING CTD CHEMICAL AND GENE VOCABULARIES
    # ========================================================

    # Chemical vocabulary (case-insensitive)
    ctd_chem_set = set(
        CTDchemicals["# ChemicalName"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Gene vocabulary (case-insensitive)
    ctd_gene_set = set(
        pd.concat([
            CTDgenes["# GeneSymbol"],
            CTDgenes["GeneName"]
        ])
        .dropna()
        .astype("string")
        .str.strip()
        .str.upper()
    )

    sample["chemical_name_clean"] = (
        sample["chemicalName"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    sample["gene_name_clean"] = (
        sample["geneSymbol"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    chem_mask = (
        sample["chemical_name_clean"].ne("")
        &
        sample["chemical_name_clean"].isin(ctd_chem_set)
    )

    gene_mask = (
        sample["gene_name_clean"].ne("")
        &
        sample["gene_name_clean"].isin(ctd_gene_set)
    )

    sample = sample[
        chem_mask & gene_mask
    ].copy()

    sample = sample.drop(
        columns=[
            "chemical_name_clean",
            "gene_name_clean"
        ]
    )

    print(
        "After CTD vocabulary filtering:",
        len(sample)
    )

    # ========================================================
    # 3. FILTER DISCUSSION SECTIONS WITH REFERENCES
    # ========================================================

    bad_discuss_patterns = [
        r"\bprevious studies\b",
        r"\breported that\b",
        r"\bet al\b",
        r"\bprevious works\b",
        r"\baccording to\b",
    ]

    discuss_pattern = "|".join(bad_discuss_patterns)

    mask_discuss_bad = (
        sample["section_type"].eq("DISCUSS")
        &
        sample["text"].str.contains(
            discuss_pattern,
            case=False,
            regex=True,
            na=False
        )
    )

    sample = sample[
        ~mask_discuss_bad
    ].copy()

    # # ========================================================
    # # 4. DROP ROWS WITH INVALID ENTITY FLAGS
    # # ========================================================

    # flag_cols = [
    #     "chemicalFlag",
    #     "relationshipWarningFlag",
    #     "prefixMissingFlag",
    #     "geneSymbolFlag"
    # ]

    # for col in flag_cols:
    #     if col not in sample.columns:
    #         raise KeyError(
    #             f"Required flag column missing: {col}"
    #         )

    # # Chemical, relationship, and prefix flags
    # # must always be zero.
    # flag_mask = pd.Series(
    #     True,
    #     index=sample.index
    # )

    # for col in [
    #     "chemicalFlag",
    #     "relationshipWarningFlag",
    #     "prefixMissingFlag"
    # ]:
    #     flag_mask &= pd.to_numeric(
    #         sample[col],
    #         errors="coerce"
    #     ).eq(0)

    # ============================================================
    # GENE SYMBOL FLAG FILTERING
    # Keep if:
    #   1. geneSymbolFlag == 0
    #   OR
    #   2. geneSymbolFlag == 1 AND high-confidence normalization
    #      successfully mapped the gene to a valid Gene ID
    # ============================================================
    
    gene_flag_valid = (
        pd.to_numeric(
            sample["geneSymbolFlag"],
            errors="coerce"
        ).eq(0)
    )
    
    if "normalization_status" in sample.columns:
    
        high_confidence_match = (
            sample["normalization_status"]
            .fillna("")
            .astype(str)
            .eq("high_confidence_normalized")
        )
    
        valid_gene_id = (
            sample["geneID"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.fullmatch(r"\d+")
            .fillna(False)
        )
    
        verified_gene = high_confidence_match & valid_gene_id
    
    else:
        verified_gene = pd.Series(False, index=sample.index)
    
    # Keep valid genes OR high-confidence normalized genes
    flag_mask &= (gene_flag_valid | verified_gene)
    # Apply all entity and relationship flag filters
    sample = sample.loc[flag_mask].copy()

    # ========================================================
    # 5. REMOVE TEXT WITH REFERENCE CITATIONS
    # ========================================================

    mask_brackets = (
        sample["text"].str.contains(
            r"\[\d+\]",
            regex=True,
            na=False
        )
    )

    sample = sample[
        ~mask_brackets
    ].copy()

    # ========================================================
    # 6. TARGETED UNCERTAINTY FILTERING
    # ========================================================

    uncertain_pattern = (
        r"\b(?:"
        r"might|maybe|perhaps|potentially|"
        r"possibly|conceivably|plausibly|"
        r"hypothesized|speculated"
        r")\b"
    )

    mask_uncertain = (
        sample["source"].str.contains(
            uncertain_pattern,
            case=False,
            regex=True,
            na=False
        )
    )

    sample = sample[
        ~mask_uncertain
    ].copy()

    # ========================================================
    # 7. DROP EXCLUDED SECTION TYPES
    # ========================================================

    excluded_sections = [
        "METHODS",
        "INTRO",
        "CONCL",
        "ABBR",
        "DISCUSS"
    ]

    sample = sample[
        ~sample["section_type"].isin(
            excluded_sections
        )
    ].copy()

    print(
        "After section filtering:",
        len(sample)
    )

    # ========================================================
    # 8. TARGETED NEGATIVE-RESULT FILTERING
    # ========================================================

    negative_patterns = [
        r"\bno significant (?:change|effect|difference)\b",
        r"\bnot statistically significant\b",
        r"\b(?:did|does) not (?:affect|alter|change)\b",
        r"\bno (?:effect|change|difference)\b",
        r"\bunchanged\b",
        r"\b(?:did|does) not induce\b",
        r"\bno observable effect\b",
        r"\bfailed to demonstrate\b",
        r"\bnot associated with\b",
        r"\bno correlation\b",
        r"\bno relationship\b",
        r"\bno impact\b",
        r"\bno influence\b",
        r"\bwas ineffective\b"
    ]

    negative_pattern = "|".join(
        negative_patterns
    )

    mask_negative = (
        sample["source"].str.contains(
            negative_pattern,
            case=False,
            regex=True,
            na=False
        )
    )

    sample = sample[
        ~mask_negative
    ].copy()

    print(
        "After negative-result filtering:",
        len(sample)
    )

    # ========================================================
    # 9. TARGETED BROAD/GENERAL STATEMENT FILTERING
    # ========================================================

    broad_patterns = [
        r"\bit is known\b",
        r"\bwidely believed\b",
        r"\bprevious research\b",
        r"\baccording to studies\b",
        r"\breported previously\b",
        r"\bnumerous studies\b",
        r"\bit has been suggested\b",
        r"\bgenerally accepted\b",
        r"\bwidely recognized\b"
    ]

    broad_pattern = "|".join(
        broad_patterns
    )

    mask_broad = (
        sample["source"].str.contains(
            broad_pattern,
            case=False,
            regex=True,
            na=False
        )
    )

    sample = sample[
        ~mask_broad
    ].copy()

    # ========================================================
    # 10. REMOVE MISSING ENTITY MAPS
    # ========================================================

    allowed_sections = [
        "FIG",
        "FIG_TABLE",
        "TABLE",
        "SUPPLEMENT"
    ]

    mask_chem_map_empty = (
        sample["chemical_map"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    mask_gene_map_empty = (
        sample["gene_map"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    mask_maps_empty = (
        mask_chem_map_empty
        |
        mask_gene_map_empty
    )

    mask_section_not_allowed = (
        ~sample["section_type"].isin(
            allowed_sections
        )
    )

    sample = sample[
        ~(
            mask_maps_empty
            &
            mask_section_not_allowed
        )
    ].copy()

    # ========================================================
    # 11. TARGETED REASONING FILTERING
    # ========================================================

    reasoning_patterns = [
        r"\bremove this relationship\b",
        r"\brelationship is invalid\b",
        r"\bunsupported relationship\b",
        r"\bno supporting evidence\b",
        r"\bchemical not found\b",
        r"\bgene not found\b",
        r"\bincorrect chemical\b",
        r"\bincorrect gene\b",
        r"\bno effect\b",
        r"\bdoes not affect\b",
    ]

    reasoning_pattern = "|".join(
        reasoning_patterns
    )

    mask_bad_reasoning = (
        sample["reasoning"].str.contains(
            reasoning_pattern,
            case=False,
            regex=True,
            na=False
        )
    )

    sample = sample[
        ~mask_bad_reasoning
    ].copy()

    # ========================================================
    # 12. EXACT ENTITY MAP VALIDATION
    # ========================================================

    excluded_map_sections_chem = [
        "TABLE",
        "SUPPLEMENT"
    ]

    excluded_map_sections_gene = [
        "TABLE",
        "SUPPLEMENT",
        "FIG",
        "FIG_TABLE"
    ]

    mask_chem_in_map = sample.apply(
        lambda r: id_in_map(
            r["chemicalID"],
            r["chemical_map"]
        ),
        axis=1
    )

    mask_gene_in_map = sample.apply(
        lambda r: id_in_map(
            r["geneID"],
            r["gene_map"]
        ),
        axis=1
    )

    mask_chem_pass = (
        mask_chem_in_map
        |
        sample["section_type"].isin(
            excluded_map_sections_chem
        )
    )

    mask_gene_pass = (
        mask_gene_in_map
        |
        sample["section_type"].isin(
            excluded_map_sections_gene
        )
    )

    sample = sample[
        mask_chem_pass
        &
        mask_gene_pass
    ].copy()

    # ========================================================
    # REMOVE RELATIONSHIPS WITHOUT EXPLICIT CTD PREFIX
    # ========================================================
    
    sample["_relationship_check"] = (
        sample["relationship"]
        .fillna("")
        .apply(normalize_relationship)
    )
    
    valid_relationship = sample["_relationship_check"].str.match(
        r"^(increase|decrease|affect)\s+\S+"
    )
    
    print(
        "Removing relationships without valid prefix:",
        (~valid_relationship).sum()
    )
    
    sample = sample.loc[valid_relationship].copy()
    sample.drop(columns=["_relationship_check"], inplace=True)

    # ========================================================
    # FIG CAPTION FILTERING
    # ========================================================
    
    sample = flag_caption_evidence(sample)
    
    is_fig = (
        sample["section_type"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
        .eq("FIG")
    )
    
    remove_fig = (
        is_fig
        & sample["caption_review_required"].fillna(False)
    )
    
    print(
        f"Removing FIG predictions based on caption: "
        f"{remove_fig.sum():,}"
    )
    
    sample = sample.loc[~remove_fig].copy()

    # Ensure relationship warnings are actually removed
    if "relationshipWarningFlag" in sample.columns:
        warning_flags = pd.to_numeric(
            sample["relationshipWarningFlag"],
            errors="coerce"
        )
    
        sample = sample.loc[
            warning_flags.eq(0)
        ].copy()
    # ========================================================
    # 13. FINAL OUTPUT
    # ========================================================

    sample = sample.reset_index(drop=True)

    print(
        f"Output length of data: {len(sample)}"
    )

    return sample

## 6. CTD matching 

In [ ]:
def match_manual_and_df(manual: pd.DataFrame, df: pd.DataFrame):

    manual = manual.copy()
    df = df.copy()

    # --------------------------------------------------
    # 1️⃣ Filter manual rows
    # --------------------------------------------------
    if "Chain?" in manual.columns:
        manual = manual[~manual["Chain?"].fillna(False)].reset_index(drop=True)

    # --------------------------------------------------
    # 2️⃣ Rename manual columns
    # --------------------------------------------------
    manual = manual.rename(columns={
        "PMID": "pmid",
        "Interaction Actions": "relationship",
        "InteractionActions": "relationship",
        "Chemical ID": "chemicalID",
        "ChemicalID": "chemicalID",
        "Gene Symbol": "geneSymbol",
        "GeneSymbol": "geneSymbol",
        "Gene ID": "geneID",
        "GeneID": "geneID"
    })

    # --------------------------------------------------
    # 3️⃣ Clean PMID
    # --------------------------------------------------
    def clean_pmid(series):
        return (
            series.astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"[^\d]", "", regex=True)
            .replace("", pd.NA)
            .astype("Int64")
        )

    manual["pmid"] = clean_pmid(manual["pmid"])
    df["pmid"] = clean_pmid(df["pmid"])

    # --------------------------------------------------
    # 4️⃣ Normalize chemicalID
    # --------------------------------------------------
    manual["chemicalID"] = (
        manual["chemicalID"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["chemicalID"] = (
        df["chemicalID"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # --------------------------------------------------
    # 5️⃣ Normalize geneID and geneSymbol
    # --------------------------------------------------
    manual["geneID"] = pd.to_numeric(
        manual["geneID"], errors="coerce"
    ).astype("Int64")

    df["geneID"] = pd.to_numeric(
        df["geneID"], errors="coerce"
    ).astype("Int64")

    manual["geneSymbol"] = (
        manual["geneSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["geneSymbol"] = (
        df["geneSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\..+$", "", regex=True)
    )

    # --------------------------------------------------
    # 6️⃣ Build gene_key: prefer geneSymbol, fall back to geneID
    # --------------------------------------------------
    def make_gene_key(row):
        sym = row["geneSymbol"]
        if pd.notna(sym) and sym not in ("", "NAN", "NA"):
            return sym
        if pd.notna(row["geneID"]):
            return str(int(row["geneID"]))
        return pd.NA

    manual["gene_key"] = manual.apply(make_gene_key, axis=1)
    manual["gene_key"] = manual["gene_key"].astype("string")

    # if 'gene_key' not in df.columns:
    #     df["gene_key"] = df.apply(make_gene_key, axis=1)
    #     df["gene_key"] = df["gene_key"].astype("string")
    # else:
    #     df["gene_key"] = df["gene_key"].astype("string")
    df["gene_key"] = (
        df.apply(make_gene_key, axis=1)
        .astype("string")
    )

    # --------------------------------------------------
    # 7️⃣ Normalize relationship
    # --------------------------------------------------
    def normalize_relationship(s):
        if pd.isna(s):
            return s
        s = str(s).lower().strip()
        s = s.replace("^", " ")
        s = s.replace("increases", "increase")
        s = s.replace("decreases", "decrease")
        s = s.replace("affects", "affect")
        s = " ".join(s.split())
        return s

    manual["relationship_norm"] = manual["relationship"].apply(normalize_relationship)

    # if 'relationship_norm' not in df.columns:
    #     df["relationship_norm"] = df["relationship"].apply(normalize_relationship)
    df["relationship_norm"] = (
        df["relationship"]
        .apply(normalize_relationship)
    )

    # --------------------------------------------------
    # 8️⃣ Deduplicate manual
    # --------------------------------------------------
    manual = manual.drop_duplicates(
        subset=["pmid", "chemicalID", "relationship", "geneSymbol"]
    ).reset_index(drop=True)

    merge_keys = ["pmid", "relationship_norm", "chemicalID", "gene_key"]

    df_deduped = df

    # --------------------------------------------------
    # 9️⃣ Overall merge
    # --------------------------------------------------
    matched = manual.merge(
        df_deduped,
        on=merge_keys,
        how="inner",
        suffixes=("_manual", "_df")
    )

    print("=" * 60)
    print("OVERALL MATCHING STATISTICS")
    print("=" * 60)
    print(f"Total matched rows:    {len(matched)}")
    print(f"Length of CTD data:    {len(manual)}")
    print(f"Overall Precision:     {len(matched) / len(df) if len(df) else 0:.4f}")
    print(f"Overall Recall:        {len(matched) / len(manual) if len(manual) else 0:.4f}")

    # --------------------------------------------------
    # 🧬 Gene Symbol Validity Statistics
    # --------------------------------------------------
    valid_gene_symbols = set(
        manual["geneSymbol"]
        .dropna()
        .loc[lambda s: ~s.isin(["", "NAN", "NA"])]
        .unique()
    )

    df_gene_symbols = df["geneSymbol"].astype(str).str.upper().str.strip()
    df_has_valid_symbol = df_gene_symbols.notna() & ~df_gene_symbols.isin(["", "NAN", "NA"])

    df["_gene_symbol_valid"]   = df_has_valid_symbol & df_gene_symbols.isin(valid_gene_symbols)
    df["_gene_symbol_invalid"] = df_has_valid_symbol & ~df_gene_symbols.isin(valid_gene_symbols)

    valid_gene_count   = df["_gene_symbol_valid"].sum()
    invalid_gene_count = df["_gene_symbol_invalid"].sum()
    no_symbol_count    = (~df_has_valid_symbol).sum()

    print(f"\n--- Gene Symbol Validity (df vs. manual) ---")
    print(f"  Valid gene symbols   (found in manual): {valid_gene_count}")
    print(f"  Invalid gene symbols (not in manual):   {invalid_gene_count}")
    print(f"  Rows with no gene symbol:               {no_symbol_count}")
    print(f"  Total df rows:                          {len(df)}")

    df.drop(columns=["_gene_symbol_valid", "_gene_symbol_invalid"], inplace=True)

    # --------------------------------------------------
    # 🧪 Chemical ID Validity Statistics
    # --------------------------------------------------
    valid_chemical_ids = set(
        manual["chemicalID"]
        .dropna()
        .loc[lambda s: ~s.isin(["", "NAN", "NA"])]
        .unique()
    )

    df_chem_ids = df["chemicalID"].astype(str).str.upper().str.strip()
    df_has_valid_chem = df_chem_ids.notna() & ~df_chem_ids.isin(["", "NAN", "NA"])

    df["_chem_valid"]   = df_has_valid_chem & df_chem_ids.isin(valid_chemical_ids)
    df["_chem_invalid"] = df_has_valid_chem & ~df_chem_ids.isin(valid_chemical_ids)

    valid_chem_count   = df["_chem_valid"].sum()
    invalid_chem_count = df["_chem_invalid"].sum()
    no_chem_count      = (~df_has_valid_chem).sum()

    print(f"\n--- Chemical ID Validity (df vs. manual) ---")
    print(f"  Valid chemical IDs   (found in manual): {valid_chem_count}")
    print(f"  Invalid chemical IDs (not in manual):   {invalid_chem_count}")
    print(f"  Rows with no chemical ID:               {no_chem_count}")
    print(f"  Total df rows:                          {len(df)}")

    df.drop(columns=["_chem_valid", "_chem_invalid"], inplace=True)

    # --------------------------------------------------
    # 🔗 Relationship Error Analysis on Valid Chem-Gene Pairs
    # --------------------------------------------------
    def split_relationship(rel):
        """Return (prefix, rel_type) for a normalized relationship string."""
        if pd.isna(rel):
            return (pd.NA, pd.NA)
        parts = str(rel).strip().split(" ", 1)
        prefix   = parts[0] if len(parts) >= 1 else pd.NA
        rel_type = parts[1] if len(parts) == 2 else pd.NA
        return (prefix, rel_type)

    manual_pair_rels = (
        manual.groupby(["pmid", "chemicalID", "gene_key"])["relationship_norm"]
        .apply(set)
        .reset_index()
        .rename(columns={"relationship_norm": "manual_rels"})
    )

    df_pair_check = df[["pmid", "chemicalID", "gene_key", "relationship_norm"]].copy()
    df_pair_check = df_pair_check.merge(manual_pair_rels, on=["pmid", "chemicalID", "gene_key"], how="left")

    has_valid_pair = df_pair_check["manual_rels"].notna()

    exact_match = has_valid_pair & df_pair_check.apply(
        lambda r: r["relationship_norm"] in r["manual_rels"] if pd.notna(r["manual_rels"]) else False,
        axis=1
    )

    wrong_rel = has_valid_pair & ~exact_match
    
    mismatch_df = df_pair_check[wrong_rel].copy()
    mismatch_df[["df_prefix", "df_type"]] = pd.DataFrame(
        mismatch_df["relationship_norm"].apply(split_relationship).tolist(),
        index=mismatch_df.index,
        columns=["df_prefix", "df_type"]
    )
    
    mismatch_expanded = mismatch_df.explode("manual_rels").copy()
    mismatch_expanded[["manual_prefix", "manual_type"]] = pd.DataFrame(
        mismatch_expanded["manual_rels"].apply(split_relationship).tolist(),
        index=mismatch_expanded.index,
        columns=["manual_prefix", "manual_type"]
    )

    def classify_mismatch(group):
        df_prefix = group["df_prefix"].iloc[0]
        df_type   = group["df_type"].iloc[0]

        prefix_matches = (group["manual_prefix"] == df_prefix).any()
        type_matches   = (group["manual_type"]   == df_type  ).any()

        if prefix_matches and not type_matches:
            return "wrong_type_only"
        elif type_matches and not prefix_matches:
            return "wrong_prefix_only"
        else:
            return "wrong_prefix_and_type"

    if mismatch_expanded.empty:
        mismatch_categories = pd.Series(dtype="object", name="error_category")
    else:
        mismatch_categories = (
            mismatch_expanded
            .groupby(mismatch_expanded.index)
            .apply(classify_mismatch)
        )
        mismatch_categories.name = "error_category"
    
    category_counts = mismatch_categories.value_counts()

    wrong_prefix_only     = category_counts.get("wrong_prefix_only",     0)
    wrong_type_only       = category_counts.get("wrong_type_only",       0)
    wrong_prefix_and_type = category_counts.get("wrong_prefix_and_type", 0)

    print(f"\n--- Relationship Error Analysis on Valid Chem-Gene Pairs ---")
    print(f"  DF rows with a valid (pmid, chemicalID, gene_key) pair: {has_valid_pair.sum()}")
    print(f"  Exact relationship match:                               {exact_match.sum()}")
    print(f"  Wrong relationship (any mismatch):                      {wrong_rel.sum()}")
    print(f"    ├─ Wrong prefix only  (correct type, wrong direction): {wrong_prefix_only}")
    print(f"    ├─ Wrong type only    (correct prefix, wrong type):    {wrong_type_only}")
    print(f"    └─ Wrong prefix AND type:                              {wrong_prefix_and_type}")

    # --------------------------------------------------
    # 🔟 Unmatched rows
    # --------------------------------------------------
    matched_keys = matched[merge_keys].drop_duplicates(
        subset=["pmid", "gene_key", "chemicalID", "relationship_norm"]
    )

    unmatched_manual = manual.merge(
        matched_keys, on=merge_keys, how="left", indicator=True
    )
    unmatched_manual = unmatched_manual[
        unmatched_manual["_merge"] == "left_only"
    ].drop(columns="_merge")

    unmatched_df = df.merge(
        matched_keys, on=merge_keys, how="left", indicator=True
    )
    unmatched_df = unmatched_df[
        unmatched_df["_merge"] == "left_only"
    ].drop(columns="_merge")

    print(f"\nUnmatched manual rows: {len(unmatched_manual)}")
    print(f"Unmatched df rows:     {len(unmatched_df)}")

    # --------------------------------------------------
    # 1️⃣1️⃣ Partial matches
    # --------------------------------------------------
    partial_keys   = ["pmid", "geneSymbol", "chemicalID"]
    extra_df_cols  = ["relationship_norm", "gene_key"]

    partial_matches = unmatched_manual.merge(
        unmatched_df[partial_keys + extra_df_cols].drop_duplicates(
            subset=["pmid", "gene_key", "chemicalID"]
        ),
        on=partial_keys,
        how="inner",
        suffixes=("_manual", "_df")
    )

    if len(partial_matches) > 0:
        print(f"\n⚠️  {len(partial_matches)} rows matched on pmid+geneSymbol+chemicalID but NOT on full merge keys")

        print("\n--- Manual relationship_norm (unmatched, partial key match) ---")
        print(partial_matches["relationship_norm_manual"].value_counts().to_string())

        print("\n--- DF relationship_norm (unmatched, partial key match) ---")
        print(partial_matches["relationship_norm_df"].value_counts().to_string())

        print("\n--- Side-by-side (Manual → DF) ---")
        paired = (
            partial_matches
            .groupby(["relationship_norm_manual", "relationship_norm_df"])
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        print(paired.to_string(index=False))

    else:
        print("\n✅ No partial matches — unmatched rows don't share pmid+geneSymbol+chemicalID")

    print("=" * 60 + "\n")

    return matched, unmatched_manual, unmatched_df, partial_matches

## 7. Load model CSVs from `refined_combined`


In [ ]:
MODEL_NAMES = [
    # Extraction
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',

    # DeepSeek validation
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    'deepseekV_deepseekE_output',

    # GPT validation
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    'gptV_gptE_output',

    # Claude validation
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    'claudeV_claudeE_output',

    # GPT-4o validation
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output',
    'gpt4oV_gpt4oE_output',
]

def load_merged(model):
    """Load one model CSV from the combined refined output folder."""
    path = REFINED_DIR / f"{model}.csv"

    if not path.is_file():
        print(f"  missing: {path}")
        return None

    df = pd.read_csv(path, low_memory=False)
    if "pmid" not in df.columns:
        raise KeyError(f"Missing pmid column in {path}")
    df["pmid"] = pd.to_numeric(df["pmid"], errors="coerce").astype("Int64")
    return df


## 8. FINAL matching

In [ ]:
manual = ctd_ixns
summary_rows = []
print("FINAL (combined refined outputs) — ORIGINAL PROCESSING ORDER")

SAVE_DIR = ROOT / "CLOVER_output_files" / "ctd_matching_combined_final"

for i, model in enumerate(MODEL_NAMES, 1):
    log(f"[{i}/{len(MODEL_NAMES)}] {model}")
    df = load_merged(model)
    if df is None:
        print(f"{model}: no files found, skipped")
        continue
    print(f"{model}: loaded {len(df):,} rows")
    raw_count = len(df)
    

    # df = drop_duplicates_(df)
    # unique_pmids = df["pmid"].dropna().unique()
    # print(f"Unique PMIDs: {len(unique_pmids)}")
    
    # dedup_count = len(df)
    
    df = update_dataframe(df)
    updated_count = len(df)
    
    df = normalize_gene_symbols(
        df,
        default_species="hsa",
        strict_conflicts=True,
        use_mirgenedb=True,
        use_hgnc=True,
        use_uniprot=True
    )

    df = update_gene_symbol_(df, ctd_genes, ctd_lookup=CTD_SYMBOL_LOOKUP)
    
    # Deduplicate after symbol normalization
    df = drop_duplicates_(df)
    
    df = filter_dataframe(df, ctd_chemicals, ctd_genes)
    filtered_count = len(df)
        

    print(model)
    matched, unmatched_manual, unmatched_df, partial_matches = match_manual_and_df(manual, df)
    
    # Distribution of relationship types matched with CTD
    print(f"\n{model} — Matched CTD Relationship Distribution")
    
    relationship_dist = (
        matched["relationship_manual"]
        .value_counts()
        .reset_index()
    )
    
    relationship_dist.columns = ["relationship_manual", "count"]
    
    print(f"Total matched rows: {len(matched):,}")
    print(f"Unique relationship types: {matched['relationship_manual'].nunique()}")
    display(relationship_dist)

    matched.to_csv(SAVE_DIR / f"{model}_matched.csv", index=False)
    unmatched_manual.to_csv(SAVE_DIR / f"{model}_unmatched_manual.csv", index=False)
    unmatched_df.to_csv(SAVE_DIR / f"{model}_unmatched_df.csv", index=False)
    partial_matches.to_csv(SAVE_DIR / f"{model}_partial_matches.csv", index=False)
    df.to_csv(SAVE_DIR / f"{model}_filtered_predictions.csv", index=False)

    tp = len(matched)
    precision = tp / filtered_count if filtered_count else 0.0
    recall = tp / len(manual) if len(manual) else 0.0
    summary_rows.append({"model": model, "raw_rows": raw_count, "after_dedup": dedup_count,
                         "after_update": updated_count, "after_filter": filtered_count,
                         "matched_rows": tp, "precision": precision, "recall": recall})
    pd.DataFrame(summary_rows).to_csv(SAVE_DIR / "summary.csv", index=False)
    print(f"STAGE CHECK: raw={raw_count:,}, dedup={dedup_count:,}, filtered={filtered_count:,}, matched={tp:,}")
    if model == "deepseek_E_output":
        print("Historical DeepSeek comparison: expected pre-filter=7893, output=2417, matched=658")
        print("Observed:", dedup_count, filtered_count, tp)
    print()

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print("Saved in", SAVE_DIR)

In [ ]:

import pandas as pd

manual = ctd_ixns
summary_rows = []

SAVE_DIR = ROOT / "CLOVER_output_files" / "ctd_matching_combined_standardized"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("STANDARDIZED — NO FILTER")

for i, model in enumerate(MODEL_NAMES, 1):
    log(f"[{i}/{len(MODEL_NAMES)}] {model}")

    df = load_merged(model)
    if df is None:
        print(f"{model}: no files found, skipped")
        continue

    raw_count = len(df)

    # Standardization
    df = update_dataframe(df)
    updated_count = len(df)

    df = normalize_gene_symbols(
        df,
        default_species="hsa",
        strict_conflicts=True,
        use_mirgenedb=True,
        use_hgnc=True,
        use_uniprot=True
    )

    df = update_gene_symbol_(
        df,
        ctd_genes,
        ctd_lookup=CTD_SYMBOL_LOOKUP
    )

    # Deduplicate, but do NOT filter
    df = df.drop_duplicates(
        subset=["pmid", "chemicalID", "geneSymbol", "relationship"],
        keep="first"
    ).copy()

    dedup_count = len(df)

    # Match the standardized predictions directly
    matched, unmatched_manual, unmatched_df, partial_matches = (
        match_manual_and_df(manual, df)
    )

    # Save all results
    matched.to_csv(SAVE_DIR / f"{model}_matched.csv", index=False)
    unmatched_manual.to_csv(SAVE_DIR / f"{model}_unmatched_manual.csv", index=False)
    unmatched_df.to_csv(SAVE_DIR / f"{model}_unmatched_df.csv", index=False)
    partial_matches.to_csv(SAVE_DIR / f"{model}_partial_matches.csv", index=False)
    df.to_csv(SAVE_DIR / f"{model}_filtered_predictions.csv", index=False)

    tp = len(matched)
    precision = tp / len(df) if len(df) else 0.0
    recall = tp / len(manual) if len(manual) else 0.0

    summary_rows.append({
        "model": model,
        "raw_rows": raw_count,
        "after_update": updated_count,
        "after_dedup": dedup_count,
        "after_filter": dedup_count,
        "matched_rows": tp,
        "precision": precision,
        "recall": recall
    })

    print(
        f"{model}: "
        f"Raw={raw_count:,} | "
        f"Standardized={dedup_count:,} | "
        f"Matched={tp:,} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f}"
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SAVE_DIR / "summary.csv", index=False)

display(summary_df)
print("Saved in:", SAVE_DIR)


### extracted matches

In [ ]:
def repair_csv(val_str: str) -> str:
    """
    Repairs CSV strings where a free-text field (e.g. geneName) may contain
    commas, causing row field counts to exceed the header field count.

    Strategy: Instead of hardcoding column positions, this function:
      1. Parses the header to get the expected column count (N).
      2. For each malformed row (too many fields), finds the LONGEST
         contiguous run of 'ambiguous' tokens — i.e. the tokens that,
         when collapsed back into one comma-joined string, produce exactly
         N fields total.
      3. Identifies which column is the free-text one by scanning the header
         for known free-text field names (e.g. 'geneName', 'explanation').
         Falls back to trying every possible merge window if no known field found.

    This means it works correctly whether the schema is:
        chemicalName, chemicalID, relationship, geneName, geneID          (5 cols)
        chemicalName, relationship, geneName, geneSymbol, agreement, reasoning  (6 cols)
        ...or any other schema where exactly one field may contain commas.
    """

    # Known free-text fields that may contain commas, in priority order
    FREE_TEXT_FIELDS = [
        'genename', 'explanation', 'reasoning', 'description',
        'caption', 'text', 'name'
    ]

    # ✅ Extract CSV blocks from markdown code fences
    # If there are multiple ```csv blocks, keep only the last one
    if '```' in val_str:
        blocks = []
        current_block = []
        in_code_block = False
        
        for line in val_str.splitlines():
            stripped = line.strip()
            if stripped.startswith('```'):
                if in_code_block:
                    # End of block
                    if current_block:
                        blocks.append('\n'.join(current_block))
                    current_block = []
                    in_code_block = False
                else:
                    # Start of block
                    in_code_block = True
            elif in_code_block:
                current_block.append(line)
        
        # If there are blocks, use the last one; otherwise use original
        if blocks:
            val_str = blocks[-1].strip()

    lines = val_str.strip().splitlines()
    if not lines:
        return val_str

    # Parse header
    try:
        header_parts = next(csv.reader([lines[0]]))
    except Exception:
        return val_str

    expected = len(header_parts)

    # Find which column index is the free-text field
    free_text_idx = None
    header_lower = [h.strip().lower() for h in header_parts]
    for field in FREE_TEXT_FIELDS:
        if field in header_lower:
            free_text_idx = header_lower.index(field)
            break

    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(header_parts)

    for line in lines[1:]:
        if not line.strip():
            continue

        # Parse the line respecting existing quotes first
        try:
            parts = next(csv.reader([line]))
        except Exception:
            # If quoted parsing fails, fall back to raw split
            parts = line.split(',')

        if len(parts) == expected:
            writer.writerow(parts)
            continue

        if len(parts) < expected:
            # Under-count — write as-is, let pandas handle/warn
            writer.writerow(parts)
            continue

        # Too many fields — need to merge some tokens
        extra = len(parts) - expected  # how many extra commas exist

        if free_text_idx is not None:
            # We know which field to merge into: collapse `extra+1` tokens
            # starting at free_text_idx into a single value
            merge_count = extra + 1
            prefix = parts[:free_text_idx]
            merged = ','.join(parts[free_text_idx: free_text_idx + merge_count])
            suffix = parts[free_text_idx + merge_count:]
            repaired = prefix + [merged] + suffix
            if len(repaired) == expected:
                writer.writerow(repaired)
                continue

        # Fallback: try every possible merge window and pick the first
        # one that produces exactly `expected` fields
        repaired = None
        for start in range(len(parts) - extra):
            end = start + extra + 1  # merge tokens[start:end]
            candidate = parts[:start] + [','.join(parts[start:end])] + parts[end:]
            if len(candidate) == expected:
                repaired = candidate
                break

        if repaired:
            writer.writerow(repaired)
        else:
            # Can't repair — write raw and let pandas warn
            writer.writerow(parts)

    return output.getvalue()


# ── Rest of the pipeline (unchanged except repair_csv is now called) ──────────

output_cols = [
    # Extraction
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',

    # DeepSeek validation
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    'deepseekV_deepseekE_output',

    # GPT validation
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    'gptV_gptE_output',

    # Claude validation
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    'claudeV_claudeE_output',

    # GPT-4o validation
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output',
    'gpt4oV_gpt4oE_output',
]

keep_cols = [
    'explanation',
    'chemicalName',
    'chemicalID',
    'relationship',
    'geneName',
    'geneID',
    'geneSymbol',
    'source',
    '_source_row',
    'agreement',
    'reasoning'
    # 'confidence'
]


def combine_csv_outputs(df, columns):
    all_dfs = {}
    extra_cols = [
        'pmid',
        'caption',
        'original_filename',
        'gene_map',
        'chemical_map',
        'section_type',
        'text',
        'table_xml'
    ]

    for col in columns:
        if col not in df.columns:
            continue

        col_dfs = []
        for idx, val in df[col].items():
            if val is None:
                continue
            if isinstance(val, pd.DataFrame):
                col_dfs.append(val)
                continue

            try:
                val_str = str(val).strip()
            except Exception:
                continue
            if not val_str:
                continue

            # ✅ Repair comma-in-free-text-field issues AND extract from code fences
            val_str = repair_csv(val_str)

            try:
                temp_df = pd.read_csv(
                    io.StringIO(val_str),
                    engine="python",
                    sep=",",
                    quotechar='"',
                    skipinitialspace=True,
                    on_bad_lines="warn"
                )
                temp_df["_source_row"] = idx
                for c in extra_cols:
                    temp_df[c] = df.at[idx, c] if c in df.columns else pd.NA
                col_dfs.append(temp_df)
            except Exception as e:
                print(f"⚠️ Failed parsing row {idx} column {col}: {e}")
                continue

        if col_dfs:
            combined_col_df = pd.concat(col_dfs, ignore_index=True, sort=False)
            for c in keep_cols:
                if c not in combined_col_df.columns:
                    combined_col_df[c] = pd.NA
            final_cols = ['_source_row'] + extra_cols + keep_cols
            combined_col_df = combined_col_df[[c for c in final_cols if c in combined_col_df.columns]]
            all_dfs[col] = combined_col_df
            print(f"done with {col} - total rows = {len(combined_col_df)}")
        else:
            print(f"done with {col} - no valid CSV")

    return all_dfs


In [ ]:
def combine_csv_outputs(df, columns):
    all_dfs = {}
    extra_cols = [
        'pmid',
        'caption',
        'original_filename',
        'gene_map',
        'chemical_map',
        'section_type',
        'text',
        'table_xml'
    ]

    for col in columns:
        if col not in df.columns:
            continue

        col_dfs = []
        for idx, val in df[col].items():
            if val is None:
                continue
            if isinstance(val, pd.DataFrame):
                col_dfs.append(val)
                continue

            try:
                val_str = str(val).strip()
            except Exception:
                continue
            if not val_str:
                continue

            # ✅ Repair comma-in-free-text-field issues AND extract from code fences
            val_str = repair_csv(val_str)

            try:
                temp_df = pd.read_csv(
                    io.StringIO(val_str),
                    engine="python",
                    sep=",",
                    quotechar='"',
                    skipinitialspace=True,
                    on_bad_lines="warn"
                )
                temp_df["_source_row"] = idx
                for c in extra_cols:
                    temp_df[c] = df.at[idx, c] if c in df.columns else pd.NA
                col_dfs.append(temp_df)
            except Exception as e:
                print(f"⚠️ Failed parsing row {idx} column {col}: {e}")
                continue

        if col_dfs:
            combined_col_df = pd.concat(col_dfs, ignore_index=True, sort=False)
            for c in keep_cols:
                if c not in combined_col_df.columns:
                    combined_col_df[c] = pd.NA
            final_cols = ['_source_row'] + extra_cols + keep_cols
            combined_col_df = combined_col_df[[c for c in final_cols if c in combined_col_df.columns]]
            all_dfs[col] = combined_col_df
            print(f"done with {col} - total rows = {len(combined_col_df)}")
        else:
            print(f"done with {col} - no valid CSV")

    return all_dfs


# Load raw outputs (NO refinement)
data = pd.read_csv(VALIDATION_CSV, low_memory=False)
combined_df = combine_csv_outputs(data, MODEL_NAMES)

# Separate output directory
RAW_SAVE_DIR = ROOT / "CLOVER_output_files" / "ctd_matching_raw_combined"
RAW_SAVE_DIR.mkdir(parents=True, exist_ok=True)

manual = ctd_ixns
summary_rows = []

print("RAW MODEL MATCHING — NO REFINEMENT")

for i, model in enumerate(MODEL_NAMES, 1):
    print(f"\n[{i}/{len(MODEL_NAMES)}] {model}")

    if model not in combined_df:
        print(f"Missing {model}, skipping.")
        continue

    # Directly from combined raw model outputs
    df = combined_df[model].copy()
    raw_count = len(df)

    # Normalize for matching
    df = update_dataframe(df)

    df = normalize_gene_symbols(
        df,
        default_species="hsa",
        strict_conflicts=True,
        use_mirgenedb=True,
        use_hgnc=True,
        use_uniprot=True
    )

    df = drop_duplicates_(df)

    # No refine() or filter_dataframe()
    filtered_count = len(df)

    # Match against CTD
    matched, unmatched_manual, unmatched_df, partial_matches = (
        match_manual_and_df(manual, df)
    )

    # Relationship distribution
    relationship_dist = (
        matched["relationship_manual"]
        .value_counts()
        .rename_axis("relationship_manual")
        .reset_index(name="count")
    )

    print(f"Matched: {len(matched):,}")
    print(f"Unique relationship types: {len(relationship_dist)}")
    display(relationship_dist)

    # Save results
    matched.to_csv(RAW_SAVE_DIR / f"{model}_matched.csv", index=False)
    unmatched_manual.to_csv(RAW_SAVE_DIR / f"{model}_unmatched_manual.csv", index=False)
    unmatched_df.to_csv(RAW_SAVE_DIR / f"{model}_unmatched_df.csv", index=False)
    partial_matches.to_csv(RAW_SAVE_DIR / f"{model}_partial_matches.csv", index=False)
    df.to_csv(RAW_SAVE_DIR / f"{model}_raw_predictions.csv", index=False)

    # Metrics
    tp = len(matched)
    precision = tp / filtered_count if filtered_count else 0.0
    recall = tp / len(manual) if len(manual) else 0.0

    summary_rows.append({
        "model": model,
        "raw_rows": raw_count,
        "after_dedup": dedup_count,
        "predictions": filtered_count,
        "matched_rows": tp,
        "precision": precision,
        "recall": recall
    })

    print(
        f"STAGE CHECK: raw={raw_count:,}, "
        f"dedup={dedup_count:,}, "
        f"predictions={filtered_count:,}, matched={tp:,}"
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RAW_SAVE_DIR / "summary.csv", index=False)

display(summary_df)
print("Saved in:", RAW_SAVE_DIR)

# Metrics

## per section (TBL SUP TXT FIG)

In [ ]:
def safe_section_match(manual, part):
    if part.empty:
        return pd.DataFrame(), manual.copy(), pd.DataFrame(), pd.DataFrame()

    try:
        return match_manual_and_df(manual, part)
    except ValueError as e:
        if "Columns must be same length as key" not in str(e):
            raise

        # Work around the empty partial-match DataFrame
        original_split = globals()["split_relationship"]

        def safe_split_relationship(value):
            return original_split(value)

        # The exception originates from empty DataFrame assignment,
        # so changing split_relationship alone will not resolve it.
        raise ValueError(
            "The matcher encountered an empty relationship-mismatch set. "
            "Its internal empty DataFrame assignment needs handling, "
            "or section matching must use an independent matcher."
        ) from e

BASE = ROOT / "CLOVER_output_files"
STAGES = {
    "raw": BASE / "ctd_matching_combined_section_raw",
    "standardized": BASE / "ctd_matching_combined_section_standardized",
    "final": BASE / "ctd_matching_combined_section_final"
}
for path in STAGES.values(): path.mkdir(parents=True, exist_ok=True)

SECTIONS = ["TABLE", "SUPP", "FIG", "TEXT"]
KEYS = ["pmid", "chemicalID", "gene_key", "relationship_norm"]
manual = ctd_ixns.copy()

def get_section(s):
    s = str(s).strip().upper()
    if s in ("TABLE", "FIG_TABLE"): return "TABLE"
    if s in ("SUPP", "SUPPLEMENT"): return "SUPP"
    if s in ("FIG", "FIGURE"): return "FIG"
    return "TEXT"

def section_dedup(df):
    df = df.copy()
    df["pmid"] = pd.to_numeric(df["pmid"], errors="coerce").astype("Int64")
    df["chemicalID"] = df["chemicalID"].fillna("").astype(str).str.upper().str.strip()
    df["relationship_norm"] = df["relationship"].apply(normalize_relationship)
    symbol = df["geneSymbol"].fillna("").astype(str).str.upper().str.strip().str.replace(r"\..+$", "", regex=True)
    gene_id = pd.to_numeric(df["geneID"], errors="coerce").astype("Int64").astype("string")
    df["gene_key"] = symbol.mask(symbol.isin(["", "NAN", "NA", "NONE", "<NA>"]), gene_id)
    return df.drop_duplicates(subset=KEYS, keep="first").reset_index(drop=True)

def prepare_stage(df, stage):
    df = df.copy()
    if stage == "raw": return df

    df = update_dataframe(df)
    df = normalize_gene_symbols(df, default_species="hsa", strict_conflicts=True,
                                use_mirgenedb=True, use_hgnc=True, use_uniprot=True)
    df = update_gene_symbol_(df, ctd_genes, ctd_lookup=CTD_SYMBOL_LOOKUP)
    return df

results = []

for i, model in enumerate(MODEL_NAMES, 1):
    print(f"\n[{i}/{len(MODEL_NAMES)}] {model}")
    original = load_merged(model)
    if original is None: continue

    for stage, save_dir in STAGES.items():
        df = prepare_stage(original, stage)
        df["section_group"] = df["section_type"].apply(get_section)

        for section in SECTIONS:
            part = df[df["section_group"] == section].copy()
            section_rows = len(part)

            # Deduplicate independently within each section
            part = section_dedup(part)
            before_filter = len(part)

            # Filtering applies only to the Final stage
            if stage == "final":
                part = filter_dataframe(part, ctd_chemicals, ctd_genes)
                part = section_dedup(part)

            n = len(part)
            if part.empty:
                matched = pd.DataFrame()
                unmatched_manual = manual.copy()
                unmatched_df = pd.DataFrame()
                partial_matches = pd.DataFrame()
            else:
                matched, unmatched_manual, unmatched_df, partial_matches = match_manual_and_df(manual, part)

            tp = len(matched)
            if tp > n:
                raise ValueError(f"{model} {stage} {section}: TP={tp} > predictions={n}")

            fp = n - tp
            total_ctd = len(matched) + len(unmatched_manual)
            p = tp / n if n else 0
            r = tp / total_ctd if total_ctd else 0
            prefix = f"{model}_{section}"

            for name, data in {
                "filtered_predictions": part,
                "matched": matched,
                "unmatched_manual": unmatched_manual,
                "unmatched_df": unmatched_df,
                "partial_matches": partial_matches
            }.items():
                data.to_csv(save_dir / f"{prefix}_{name}.csv", index=False)

            results.append({
                "Stage": stage, "Model": model, "Section": section,
                "Section_Rows": section_rows, "After_Dedup": before_filter,
                "Final_Predictions": n, "CTD_Total": total_ctd,
                "TP": tp, "FP": fp,
                "Precision": round(p, 6), "Recall": round(r, 6)
            })

            print(f"{stage.upper():12} {section:5} | Rows={section_rows:,} | "
                  f"Pred={n:,} | TP={tp:,} | FP={fp:,} | P={p:.4f} | R={r:.4f}")

section_metrics = pd.DataFrame(results)

for stage, save_dir in STAGES.items():
    sub = section_metrics[section_metrics["Stage"] == stage]
    sub.to_csv(save_dir / "section_metrics_summary.csv", index=False)

display(section_metrics)

comparison = section_metrics.pivot_table(
    index=["Model", "Section"],
    columns="Stage",
    values=["TP", "Precision", "Recall"],
    aggfunc="first"
)
display(comparison)

section_metrics.to_csv(BASE / "section_metrics_all_stages.csv", index=False)
print("Saved all section-level matching results.")

## per pmid

In [ ]:
import pandas as pd

SAVE_DIR = ROOT / "CLOVER_output_files" / "ctd_matching_combined_final"

pm_pmc_dict = {
    '9118903': '1469763', '15154616': '2275409', '15811834': '1278483',
    '15953866': '2782200', '16393664': '1332662', '17118201': '1676006',
    '17183729': '1762369', '17524151': '1890552', '18280460': '2279149',
    '18299717': '2243244', '19114014': '2631511', '19617202': '3136680',
    '19654922': '2717139', '20801540': '2987498', '21294877': '3042928',
    '22110480': '3205718', '22112310': '3279445', '22228119': '3349910',
    '22562770': '3440069', '22571318': '3485091', '22747577': '3489685',
    '22808131': '3392256', '23006841': '3488501', '23349674': '3551951',
    '23568217': '3620667', '24039778': '3769399', '24403256': '3892387',
    '24413757': '3907846', '25361045': '4245613', '25859307': '4387225',
    '26039251': '4454644', '26492236': '4632778', '26714306': '4695093',
    '27153807': '4859981', '27397436': '5298042', '27517156': '5295416',
    '27736846': '5129817', '28637469': '5480112', '29045497': '5646803',
    '29380240': '6060758', '30104959': '6077207', '30114287': '6095593',
    '31064381': '6503545', '31378678': '6706627', '32298299': '7161975',
    '34908893': '8640676', '36638633': '10016736', '37047793': '10095583',
    '25100957': '4104784', '19889203': '3003513'
}

def clean_pmid(s):
    return s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

def section_group(s):
    s = str(s).strip().upper()
    if s in ("TABLE", "FIG_TABLE"): return "Tbl"
    if s in ("SUPP", "SUPPLEMENT"): return "Sup"
    if s in ("FIG", "FIGURE"): return "Fig"
    return "Txt"

# Load Claude-E matching results
matched = pd.read_csv(SAVE_DIR / "claude_E_output_matched.csv", low_memory=False)
filtered = pd.read_csv(SAVE_DIR / "claude_E_output_filtered_predictions.csv", low_memory=False)

# Load original processed components from validation CSV
meta = pd.read_csv(VALIDATION_CSV, low_memory=False)

matched["PMID"] = clean_pmid(matched["pmid"])
filtered["PMID"] = clean_pmid(filtered["pmid"])
meta["PMID"] = clean_pmid(meta["pmid"])

# Count original processed components per article (not filtered predictions)
meta["Component"] = meta["section_type"].apply(section_group)
components = pd.crosstab(meta["PMID"], meta["Component"]).reindex(
    columns=["Txt", "Tbl", "Fig", "Sup"], fill_value=0
).reset_index()
components["Sec."] = components[["Txt", "Tbl", "Fig", "Sup"]].sum(axis=1)

# CTD reference counts
manual = ctd_ixns.copy()
manual["PMID"] = clean_pmid(manual["pmid"])

ctd_counts = manual.groupby("PMID").size().rename("CTD")
ext_counts = filtered.groupby("PMID").size().rename("Ext")
match_counts = matched.groupby("PMID").size().rename("Match")

# Build supplementary table for all 50 articles
claude_table = pd.DataFrame({"PMID": list(pm_pmc_dict.keys())})
claude_table["PMCID"] = claude_table["PMID"].map(
    lambda x: f"PMC{pm_pmc_dict[x]}"
)

claude_table = claude_table.merge(components, on="PMID", how="left")
claude_table = claude_table.merge(ctd_counts, on="PMID", how="left")
claude_table = claude_table.merge(ext_counts, on="PMID", how="left")
claude_table = claude_table.merge(match_counts, on="PMID", how="left")

for col in ["CTD", "Ext", "Match", "Sec.", "Txt", "Tbl", "Fig", "Sup"]:
    claude_table[col] = claude_table[col].fillna(0).astype(int)

claude_table["P"] = (claude_table["Match"] / claude_table["Ext"].replace(0, float("nan"))).round(3)
claude_table["R"] = (claude_table["Match"] / claude_table["CTD"].replace(0, float("nan"))).round(3)

claude_table = claude_table[
    ["PMID", "PMCID", "Sec.", "CTD", "Ext", "Match", "P", "R", "Txt", "Tbl", "Fig", "Sup"]
].sort_values(["R", "P"], ascending=False, na_position="last").reset_index(drop=True)

claude_table.to_csv(SAVE_DIR / "claude_E_supplement_table.csv", index=False)
display(claude_table)

print(f"Articles: {len(claude_table)}")
print(f"CTD: {claude_table['CTD'].sum():,} | Extracted: {claude_table['Ext'].sum():,} | Matched: {claude_table['Match'].sum():,}")
print(f"Processed components: {claude_table['Sec.'].sum():,}")

## TP/FP analysis

In [ ]:
import pandas as pd

SAVE_DIR = ROOT / "CLOVER_output_files" / "ctd_matching_combined_final"

PAIRS = {
    "DeepSeek": ("deepseek_E_output", "deepseekV_deepseekE_output"),
    "GPT-4.1": ("gpt_E_output", "gptV_gptE_output"),
    "Claude": ("claude_E_output", "claudeV_claudeE_output"),
    "GPT-4o": ("gpt4o_E_output", "gpt4oV_gpt4oE_output")
}

KEYS = ["pmid", "chemicalID", "gene_key", "relationship_norm"]

def normalize_rel(s):
    if pd.isna(s): return ""
    s = str(s).lower().strip().replace("^", " ")
    for a, b in [("increases", "increase"), ("decreases", "decrease"), ("affects", "affect")]:
        s = s.replace(a, b)
    return " ".join(s.split())

def interaction_keys(df):
    df = df.copy()
    df["pmid"] = df["pmid"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    df["chemicalID"] = df["chemicalID"].fillna("").astype(str).str.upper().str.strip()
    gene = df["geneSymbol"].fillna("").astype(str).str.upper().str.strip().str.replace(r"\..+$", "", regex=True)
    ids = pd.to_numeric(df["geneID"], errors="coerce").astype("Int64").astype("string")
    df["gene_key"] = gene.mask(gene.isin(["", "NAN", "NA"]), ids)
    df["relationship_norm"] = df["relationship"].apply(normalize_rel)
    return set(map(tuple, df[KEYS].drop_duplicates().itertuples(index=False, name=None)))

def load_stage(model):
    pred = pd.read_csv(SAVE_DIR / f"{model}_filtered_predictions.csv", low_memory=False)
    matched = pd.read_csv(SAVE_DIR / f"{model}_matched.csv", low_memory=False)
    all_keys = interaction_keys(pred)

    # Matched files contain manual/df suffixes; use the prediction-side identity.
    matched = matched.rename(columns={
        "geneSymbol_df": "geneSymbol", "geneID_df": "geneID",
        "relationship_df": "relationship"
    })
    tp = interaction_keys(matched) if len(matched) else set()
    fp = all_keys - tp

    if not tp.issubset(all_keys):
        raise ValueError(f"{model}: matched interactions missing from predictions")

    return tp, fp, len(pred)

rows = []

for model, (extractor, validator) in PAIRS.items():
    tp_e, fp_e, n_e = load_stage(extractor)
    tp_v, fp_v, n_v = load_stage(validator)

    p_e = len(tp_e) / n_e if n_e else 0
    p_v = len(tp_v) / n_v if n_v else 0

    # Same CTD denominator in both stages
    delta_p = (p_v - p_e) * 100
    delta_r = (len(tp_v) - len(tp_e)) / len(ctd_ixns) * 100

    rows.append({
        "Model": model,
        "TP Ret.": len(tp_e & tp_v),
        "TP Rem.": len(tp_e - tp_v),
        "TP Gain": len(tp_v - tp_e),
        "FP Ret.": len(fp_e & fp_v),
        "FP Rem.": len(fp_e - fp_v),
        "FP Gain": len(fp_v - fp_e),
        "ΔP (%)": round(delta_p, 2),
        "ΔR (%)": round(delta_r, 2)
    })

validation_transitions = pd.DataFrame(rows)
display(validation_transitions)

validation_transitions.to_csv(
    SAVE_DIR / "same_model_validation_transitions.csv", index=False
)

## Gene normalization

In [ ]:
import pandas as pd
from collections import Counter

BASE = ROOT / "CLOVER_output_files"
FINAL_DIR = BASE / "ctd_matching_combined_final"
SAVE_DIR = BASE / "gene_normalization_metrics"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "DeepSeek-E": "deepseek_E_output",
    "GPT-4.1-E": "gpt_E_output",
    "Claude-E": "claude_E_output",
    "GPT-4o-E": "gpt4o_E_output",
    "DS-V/GPT-4.1-E": "deepseekV_gptE_output",
    "DS-V/GPT-4o-E": "deepseekV_gpt4oE_output",
    "DS-V/Claude-E": "deepseekV_claudeE_output",
    "DS-V/DeepSeek-E": "deepseekV_deepseekE_output",
    "GPT-4.1-V/DeepSeek-E": "gptV_deepseekE_output",
    "GPT-4.1-V/Claude-E": "gptV_claudeE_output",
    "GPT-4.1-V/GPT-4.1-E": "gptV_gptE_output",
    "Cl-V/DeepSeek-E": "claudeV_deepseekE_output",
    "Cl-V/GPT-4.1-E": "claudeV_gptE_output",
    "Cl-V/GPT-4o-E": "claudeV_gpt4oE_output",
    "Cl-V/Claude-E": "claudeV_claudeE_output",
    "GPT-4o-V/DeepSeek-E": "gpt4oV_deepseekE_output",
    "GPT-4o-V/Claude-E": "gpt4oV_claudeE_output",
    "GPT-4o-V/GPT-4o-E": "gpt4oV_gpt4oE_output"
}

GENE_COLS = ["geneSymbol", "geneID", "geneName"]

DATABASES = [
    "MirGeneDB",
    "NCBI Gene",
    "HGNC",
    "UniProt",
    "Combined NCBI/HGNC",
    "Unattributed"
]

SIGNATURE_COLS = [
    "pmid",
    "chemicalID",
    "geneSymbol",
    "geneID",
    "relationship",
    "section_type"
]


# ------------------------------------------------------------
# Helper: Determine whether gene information changed
# ------------------------------------------------------------

def changed(before, after):
    a = (
        before[GENE_COLS]
        .fillna("")
        .astype(str)
        .apply(lambda col: col.str.strip())
    )

    b = (
        after[GENE_COLS]
        .fillna("")
        .astype(str)
        .apply(lambda col: col.str.strip())
    )

    return a.ne(b).any(axis=1)


# ------------------------------------------------------------
# Helper: Attribute external normalization to database
# ------------------------------------------------------------

def database_category(row):
    db = str(row.get("gene_resolution_database", "")).lower()
    status = str(row.get("gene_symbol_resolution", "")).lower()

    if "both support alias" in db or ("ncbi" in db and "hgnc" in db):
        return "Combined NCBI/HGNC"

    if "mirgenedb" in db or status == "mapped_mirgenedb":
        return "MirGeneDB"

    if "uniprot" in db or "uniprot" in status:
        return "UniProt"

    if "hgnc" in db:
        return "HGNC"

    if "ncbi gene" in db or status in (
        "mapped_official",
        "mapped_name_official"
    ):
        return "NCBI Gene"

    return "Unattributed"


# ------------------------------------------------------------
# Helper: Normalize identifiers for dataset comparison
#
# Fixes:
#   2539.0 == 2539
#   18299717.0 == 18299717
# ------------------------------------------------------------

def normalize_numeric_identifier(series):
    numeric = pd.to_numeric(series, errors="coerce")

    def format_value(value):
        if pd.isna(value):
            return ""

        if float(value).is_integer():
            return str(int(value))

        return str(value)

    normalized = numeric.map(format_value)

    # Preserve nonnumeric identifiers rather than converting
    # them to empty strings.
    original = series.fillna("").astype(str).str.strip()

    normalized = normalized.where(
        numeric.notna(),
        original
    )

    return normalized


def normalized_signatures(df):
    missing = [
        col for col in SIGNATURE_COLS
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing signature columns: {missing}"
        )

    values = df[SIGNATURE_COLS].copy()

    # Normalize numeric identifiers consistently
    values["geneID"] = normalize_numeric_identifier(
        values["geneID"]
    )

    values["pmid"] = normalize_numeric_identifier(
        values["pmid"]
    )

    # Normalize remaining string columns
    for col in [
        "chemicalID",
        "geneSymbol",
        "relationship",
        "section_type"
    ]:
        values[col] = (
            values[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.upper()
        )

    return list(
        values.itertuples(index=False, name=None)
    )


def signatures(df):
    return Counter(normalized_signatures(df))


# ------------------------------------------------------------
# Helper: Compare reconstruction with combined_final
# ------------------------------------------------------------

def print_differences(model, retained, final_saved):

    reconstructed = signatures(retained)
    saved = signatures(final_saved)

    missing = saved - reconstructed
    extra = reconstructed - saved

    print(f"\n{'=' * 65}")
    print(f"DATASET COMPARISON: {model}")
    print(f"{'=' * 65}")

    print(f"Reconstructed rows: {len(retained):,}")
    print(f"Saved final rows:   {len(final_saved):,}")
    print(f"Missing rows:       {sum(missing.values()):,}")
    print(f"Extra rows:         {sum(extra.values()):,}")

    if not missing and not extra:
        print("PASS: Reconstructed data matches combined_final")
        return True

    print("\nFirst 10 rows missing from reconstruction:")

    for key, count in list(missing.items())[:10]:
        print(f"  Count={count} | {key}")

    print("\nFirst 10 extra reconstructed rows:")

    for key, count in list(extra.items())[:10]:
        print(f"  Count={count} | {key}")

    # Save mismatches for debugging
    missing_rows = [
        dict(zip(SIGNATURE_COLS, key), Count=count)
        for key, count in missing.items()
    ]

    extra_rows = [
        dict(zip(SIGNATURE_COLS, key), Count=count)
        for key, count in extra.items()
    ]

    pd.DataFrame(
        missing_rows,
        columns=SIGNATURE_COLS + ["Count"]
    ).to_csv(
        SAVE_DIR / f"{model}_missing_from_reconstruction.csv",
        index=False
    )

    pd.DataFrame(
        extra_rows,
        columns=SIGNATURE_COLS + ["Count"]
    ).to_csv(
        SAVE_DIR / f"{model}_extra_in_reconstruction.csv",
        index=False
    )

    return False


# ------------------------------------------------------------
# Main processing
# ------------------------------------------------------------

rows = []
audits = []
failed_models = []

print("=" * 70)
print("GENE NORMALIZATION — COMBINED FINAL")
print("=" * 70)

for i, (label, model) in enumerate(MODELS.items(), 1):

    print(f"\n[{i}/{len(MODELS)}] Processing {label}")

    final_path = (
        FINAL_DIR / f"{model}_filtered_predictions.csv"
    )

    if not final_path.exists():
        print(f"ERROR: Missing saved final file: {final_path}")
        failed_models.append(model)
        continue

    # Load actual saved combined_final predictions
    final_saved = pd.read_csv(
        final_path,
        low_memory=False
    )

    # Load original merged model predictions to recreate
    # normalization history
    raw = load_merged(model)

    if raw is None:
        print(f"ERROR: Missing original output for {model}")
        failed_models.append(model)
        continue

    # --------------------------------------------------------
    # Stage 1: Initial preprocessing
    # --------------------------------------------------------

    before = update_dataframe(raw).reset_index(drop=True)

    # --------------------------------------------------------
    # Stage 2: External gene normalization
    # --------------------------------------------------------

    external = normalize_gene_symbols(
        before.copy(),
        default_species="hsa",
        strict_conflicts=True,
        use_mirgenedb=True,
        use_hgnc=True,
        use_uniprot=True
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Stage 3: Custom CTD gene normalization
    # --------------------------------------------------------

    ctd = update_gene_symbol_(
        external.copy(),
        ctd_genes,
        ctd_lookup=CTD_SYMBOL_LOOKUP
    ).reset_index(drop=True)

    if not (
        len(before) == len(external) == len(ctd)
    ):
        print(
            f"ERROR: Normalization changed row counts "
            f"for {model}"
        )
        failed_models.append(model)
        continue

    # --------------------------------------------------------
    # Track modifications
    # --------------------------------------------------------

    external_changed = changed(
        before,
        external
    )

    ctd_changed = changed(
        external,
        ctd
    )

    categories = external.apply(
        database_category,
        axis=1
    )

    # Preserve original row identity
    ctd["_audit_id"] = range(len(ctd))

    # --------------------------------------------------------
    # Stage 4: Deduplication
    # --------------------------------------------------------

    retained = drop_duplicates_(ctd)

    # --------------------------------------------------------
    # Stage 5: Final filtering
    # --------------------------------------------------------

    retained = filter_dataframe(
        retained,
        ctd_chemicals,
        ctd_genes
    ).copy()

    if "_audit_id" not in retained.columns:
        raise ValueError(
            f"{model}: Filtering removed _audit_id"
        )

    # --------------------------------------------------------
    # Verify reconstructed rows against saved combined_final
    # --------------------------------------------------------

    matches_final = print_differences(
        model,
        retained,
        final_saved
    )

    if not matches_final:
        print(
            f"SKIPPING {model}: reconstruction does not "
            "match saved combined_final predictions."
        )
        failed_models.append(model)
        continue

    # --------------------------------------------------------
    # Restrict normalization metrics to final retained rows
    # --------------------------------------------------------

    ids = (
        retained["_audit_id"]
        .astype(int)
        .to_numpy()
    )

    detail = pd.DataFrame({
        "Model": label,
        "Original_Row": ids,

        "Original_Symbol":
            before.iloc[ids]["geneSymbol"].to_numpy(),

        "External_Symbol":
            external.iloc[ids]["geneSymbol"].to_numpy(),

        "CTD_Symbol":
            ctd.iloc[ids]["geneSymbol"].to_numpy(),

        "External_Modified":
            external_changed.iloc[ids].to_numpy(),

        "CTD_Modified":
            ctd_changed.iloc[ids].to_numpy(),

        "Database":
            categories.iloc[ids].to_numpy(),

        "Resolution_Status":
            external.iloc[ids][
                "gene_symbol_resolution"
            ].to_numpy()
    })

    # --------------------------------------------------------
    # Count database modifications
    # --------------------------------------------------------

    ext_mask = detail["External_Modified"]

    counts = {
        name: int(
            (
                ext_mask &
                detail["Database"].eq(name)
            ).sum()
        )
        for name in DATABASES
    }

    rows.append({
        "Model": label,
        "Rows": len(final_saved),
        "Custom CTD": int(
            detail["CTD_Modified"].sum()
        ),
        "MirGeneDB": counts["MirGeneDB"],
        "NCBI Gene": counts["NCBI Gene"],
        "HGNC": counts["HGNC"],
        "UniProt": counts["UniProt"]
    })

    audits.append(detail)

    print(
        f"\nFINAL COUNTS — {label}\n"
        f"Rows: {len(final_saved):,}\n"
        f"Custom CTD: {detail['CTD_Modified'].sum():,}\n"
        f"MirGeneDB: {counts['MirGeneDB']:,}\n"
        f"NCBI Gene: {counts['NCBI Gene']:,}\n"
        f"HGNC: {counts['HGNC']:,}\n"
        f"UniProt: {counts['UniProt']:,}\n"
        f"Combined NCBI/HGNC: "
        f"{counts['Combined NCBI/HGNC']:,}\n"
        f"Unattributed: "
        f"{counts['Unattributed']:,}"
    )


# ------------------------------------------------------------
# Build and save final tables
# ------------------------------------------------------------

gene_normalization_table = pd.DataFrame(
    rows,
    columns=[
        "Model",
        "Rows",
        "Custom CTD",
        "MirGeneDB",
        "NCBI Gene",
        "HGNC",
        "UniProt"
    ]
)

gene_normalization_audit = (
    pd.concat(audits, ignore_index=True)
    if audits
    else pd.DataFrame()
)

gene_normalization_table.to_csv(
    SAVE_DIR / "gene_normalization_table4_final.csv",
    index=False
)

gene_normalization_audit.to_csv(
    SAVE_DIR / "gene_normalization_audit_final.csv",
    index=False
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

display(gene_normalization_table)

print(
    f"\nSuccessfully verified: "
    f"{len(rows)}/{len(MODELS)} models"
)

if failed_models:
    print("\nModels requiring investigation:")

    for model in failed_models:
        print(" -", model)

    print(
        "\nIMPORTANT: Table 4 is incomplete. "
        "Do not use the aggregate table until all "
        "models pass verification."
    )

else:
    print("\nAll models verified successfully.")

print("\nSaved to:", SAVE_DIR)

## Relationship analysis

In [ ]:
import pandas as pd

SAVE_DIR = ROOT / "CLOVER_output_files" / "ctd_matching_combined_final"

MODELS = {
    "DeepSeek-E": "deepseek_E_output",
    "GPT-4.1-E": "gpt_E_output",
    "Claude-E": "claude_E_output",
    "GPT-4o-E": "gpt4o_E_output",
    "DS-V/DS-E": "deepseekV_deepseekE_output",
    "DS-V/GPT-E": "deepseekV_gptE_output",
    "DS-V/GPT-4o-E": "deepseekV_gpt4oE_output",
    "DS-V/Cl-E": "deepseekV_claudeE_output",
    "GPT-4.1-V/DS-E": "gptV_deepseekE_output",
    "GPT-4.1-V/GPT-E": "gptV_gptE_output",
    "GPT-4.1-V/Cl-E": "gptV_claudeE_output",
    "Cl-V/DS-E": "claudeV_deepseekE_output",
    "Cl-V/GPT-E": "claudeV_gptE_output",
    "Cl-V/GPT-4o-E": "claudeV_gpt4oE_output",
    "Cl-V/Cl-E": "claudeV_claudeE_output",
    "GPT-4o-V/DS-E": "gpt4oV_deepseekE_output",
    "GPT-4o-V/Cl-E": "gpt4oV_claudeE_output",
    "GPT-4o-V/GPT-4o-E": "gpt4oV_gpt4oE_output"
}

REL_TYPES = {
    "increase expression": "IncExp",
    "decrease expression": "DecExp",
    "decrease activity": "DecAct",
    "affect expression": "AffExp",
    "increase secretion": "IncSecr",
    "increase activity": "IncAct",
    "affect response to substance": "AffResp",
    "decrease response to substance": "DecResp",
    "increase cleavage": "IncCleave"
}
COLS = list(REL_TYPES.values()) + ["Other"]

def normalize_rel(x):
    x = str(x).lower().strip().replace("^", " ")
    x = " ".join(x.split())
    words = x.split()
    if words:
        words[0] = {"increases": "increase", "decreases": "decrease", "affects": "affect"}.get(words[0], words[0])
    return " ".join(words)

def distribution(df, column, model):
    rel = df[column].dropna().apply(normalize_rel)
    cats = rel.map(REL_TYPES).fillna("Other")
    counts = cats.value_counts().reindex(COLS, fill_value=0)
    total = len(cats)
    count_row = {"Model": model, "Total": total, **counts.to_dict()}
    pct_row = {"Model": model, **(counts / total * 100).round(2).to_dict()} if total else {"Model": model, **dict.fromkeys(COLS, 0)}
    return count_row, pct_row

count_rows, pct_rows = [], []

# CTD reference relationship distribution
ctd_col = next(c for c in ["relationship", "InteractionActions", "Interaction"] if c in ctd_ixns.columns)
c, p = distribution(ctd_ixns, ctd_col, "CTD")
count_rows.append(c)
pct_rows.append(p)

# Matched interactions for each model
for label, model in MODELS.items():
    matched = pd.read_csv(SAVE_DIR / f"{model}_matched.csv", low_memory=False)
    rel_col = next(c for c in ["relationship_norm", "relationship_manual", "relationship", "relationship_df"] if c in matched.columns)
    c, p = distribution(matched, rel_col, label)
    count_rows.append(c)
    pct_rows.append(p)
    print(f"{label}: {c['Total']:,} matched interactions")

relationship_counts = pd.DataFrame(count_rows)
relationship_percentages = pd.DataFrame(pct_rows)

display(relationship_counts)
display(relationship_percentages)

relationship_counts.to_csv(SAVE_DIR / "relationship_distribution_counts.csv", index=False)
relationship_percentages.to_csv(SAVE_DIR / "relationship_distribution_percentages.csv", index=False)

# Generate pgfplotstable data for your LaTeX figure
print("\n\\pgfplotstableread[col sep=&, header=true]{%")
print("&".join(["Model"] + COLS))

for _, row in relationship_percentages.iterrows():
    values = [row["Model"]] + [f"{row[col]:.2f}" for col in COLS]
    print("&".join(values))

print("}\\datatable")

In [ ]:
import pandas as pd

SAVE_DIR = ROOT / "CLOVER_output_files" / "ctd_matching_combined_final"
MODEL = "claude_E_output"

pred = pd.read_csv(SAVE_DIR / f"{MODEL}_filtered_predictions.csv", low_memory=False)
matched = pd.read_csv(SAVE_DIR / f"{MODEL}_matched.csv", low_memory=False)
manual = ctd_ixns.copy()

def norm_rel(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower().replace("^", " ")
    words = s.split()
    if words:
        words[0] = {"increase":"increases", "decrease":"decreases", "affect":"affects"}.get(words[0], words[0])
    return " ".join(words)

manual_col = next(c for c in ["relationship", "InteractionActions", "Interaction"] if c in manual.columns)
matched_col = next(c for c in ["relationship_manual", "relationship", "relationship_norm"] if c in matched.columns)

ctd_counts = manual[manual_col].map(norm_rel).value_counts()
pred_counts = pred["relationship"].map(norm_rel).value_counts()
tp_counts = matched[matched_col].map(norm_rel).value_counts()

relationships = list(ctd_counts.index) + [r for r in pred_counts.index if r not in ctd_counts.index]
rows = []

for rel in relationships:
    ctd = int(ctd_counts.get(rel, 0))
    claude = int(pred_counts.get(rel, 0))
    tp = int(tp_counts.get(rel, 0))
    unmatched = claude - tp

    if unmatched < 0:
        raise ValueError(f"{rel}: TP={tp} exceeds Claude predictions={claude}")

    rows.append({
        "Relationship": rel + (" †" if ctd == 0 else ""),
        "CTD": ctd,
        "Claude": claude,
        "TP": tp,
        "Unmatched": unmatched,
        "P (%)": round(tp / claude * 100, 1) if claude else None,
        "R (%)": round(tp / ctd * 100, 1) if ctd else None
    })

table3 = pd.DataFrame(rows)
table3["_group"] = table3["CTD"].eq(0).astype(int)
table3 = table3.sort_values(
    ["_group", "CTD", "Claude", "Relationship"],
    ascending=[True, False, False, True]
).drop(columns="_group").reset_index(drop=True)

total_ctd = table3["CTD"].sum()
total_claude = table3["Claude"].sum()
total_tp = table3["TP"].sum()

total_row = pd.DataFrame([{
    "Relationship": "Total",
    "CTD": total_ctd,
    "Claude": total_claude,
    "TP": total_tp,
    "Unmatched": total_claude - total_tp,
    "P (%)": round(total_tp / total_claude * 100, 1) if total_claude else None,
    "R (%)": round(total_tp / total_ctd * 100, 1) if total_ctd else None
}])

table3 = pd.concat([table3, total_row], ignore_index=True)
table3.to_csv(SAVE_DIR / "claude_E_relationship_table3.csv", index=False)

display(table3)
print(f"CTD={total_ctd:,} | Claude={total_claude:,} | TP={total_tp:,} | Unmatched={total_claude-total_tp:,}")

## Figure 8 

In [ ]:

import pandas as pd

BASE = ROOT / "CLOVER_output_files"

DIRS = {
    "standardized_filtered": BASE / "ctd_matching_combined_final",
    "standardized_no_filter": BASE / "ctd_matching_combined_standardized"
}

MODEL_ORDER = [
    "deepseek_E_output", "gpt_E_output", "claude_E_output", "gpt4o_E_output",
    "deepseekV_gptE_output", "deepseekV_gpt4oE_output",
    "deepseekV_claudeE_output", "deepseekV_deepseekE_output",
    "gptV_deepseekE_output", "gptV_claudeE_output", "gptV_gptE_output",
    "claudeV_deepseekE_output", "claudeV_gptE_output",
    "claudeV_gpt4oE_output", "claudeV_claudeE_output",
    "gpt4oV_deepseekE_output", "gpt4oV_claudeE_output",
    "gpt4oV_gpt4oE_output"
]

MODEL_LABELS = [
    "DeepSeek-E", "GPT-4.1-E", "Claude-E", "GPT-4o-E",
    "DS-V/GPT-E", "DS-V/GPT-4o-E", "DS-V/Cl-E", "DS-V/DS-E",
    "GPT-4.1-V/DS-E", "GPT-4.1-V/Cl-E", "GPT-4.1-V/GPT-E",
    "Cl-V/DS-E", "Cl-V/GPT-E", "Cl-V/GPT-4o-E", "Cl-V/Cl-E",
    "GPT-4o-V/DS-E", "GPT-4o-V/Cl-E", "GPT-4o-V/GPT-4o-E"
]

LABEL_MAP = dict(zip(MODEL_ORDER, MODEL_LABELS))
SECTIONS = ["TABLE", "SUPP", "FIG", "TEXT"]

XPOS = [
    1.00, 2.35, 3.70, 5.05, 6.40, 7.75,
    9.10, 10.45, 11.80, 13.15, 14.50, 15.85,
    17.20, 18.55, 19.90, 21.25, 22.60, 23.95
]

def get_section(s):
    s = str(s).strip().upper()
    if s in ("TABLE", "FIG_TABLE"):
        return "TABLE"
    if s in ("SUPP", "SUPPLEMENT"):
        return "SUPP"
    if s in ("FIG", "FIGURE"):
        return "FIG"
    return "TEXT"

def read_file(folder, model, kind):
    path = folder / f"{model}_{kind}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    try:
        return pd.read_csv(path, low_memory=False)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

def normalize_col(s):
    return (
        s.fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )

def normalize_rel(s):
    return (
        normalize_col(s)
        .str.replace("^", " ", regex=False)
        .str.replace("INCREASES", "INCREASE", regex=False)
        .str.replace("DECREASES", "DECREASE", regex=False)
        .str.replace("AFFECTS", "AFFECT", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )

def partial_key(df, from_partial=False):
    # Match saved partial-match records to saved unmatched predictions.
    # This is only for identifying their sections, not re-running CTD matching.
    gene_col = "geneSymbol"
    rel_col = "relationship_norm"

    if from_partial:
        rel_col = "relationship_norm_df"

    if rel_col not in df.columns:
        rel_col = "relationship_df" if from_partial else "relationship"

    return pd.Series(
        list(zip(
            normalize_col(df["pmid"]),
            normalize_col(df["chemicalID"]),
            normalize_col(df[gene_col]),
            normalize_rel(df[rel_col])
        )),
        index=df.index
    )

def calculate(folder, dataset):
    rows = []

    for model in MODEL_ORDER:
        matched = read_file(folder, model, "matched")
        unmatched = read_file(folder, model, "unmatched_df")
        partial = read_file(folder, model, "partial_matches")

        # Correct: directly count rows from saved matched file
        if not matched.empty:
            matched["Section"] = matched["section_type"].apply(get_section)
            correct_counts = matched["Section"].value_counts()
        else:
            correct_counts = pd.Series(dtype=int)

        # Partial: locate saved partial matches within unmatched predictions
        if not unmatched.empty:
            unmatched["Section"] = unmatched["section_type"].apply(get_section)

            if not partial.empty:
                partial_keys = set(partial_key(partial, from_partial=True))
                unmatched_keys = partial_key(unmatched)
                unmatched["Status"] = unmatched_keys.isin(partial_keys).map(
                    {True: "Partial_Match", False: "Incorrect"}
                )
            else:
                unmatched["Status"] = "Incorrect"

            status_counts = unmatched.groupby(["Section", "Status"]).size()
        else:
            status_counts = pd.Series(dtype=int)

        for section in SECTIONS:
            correct = int(correct_counts.get(section, 0))
            partial_count = int(
                status_counts.get((section, "Partial_Match"), 0)
            )
            incorrect = int(
                status_counts.get((section, "Incorrect"), 0)
            )

            rows.append({
                "Dataset": dataset,
                "Model": model,
                "Label": LABEL_MAP[model],
                "Section": section,
                "Correct": correct,
                "Partial_Match": partial_count,
                "Incorrect": incorrect,
                "Total": correct + partial_count + incorrect
            })

        total_partial = (
            int((unmatched["Status"] == "Partial_Match").sum())
            if not unmatched.empty else 0
        )

        print(
            f"{dataset} | {model}: "
            f"Correct={len(matched):,}, "
            f"Partial={total_partial:,}, "
            f"Incorrect={len(unmatched) - total_partial:,}"
        )

    return pd.DataFrame(rows)

# Calculate from saved results only
counts = pd.concat(
    [calculate(folder, name) for name, folder in DIRS.items()],
    ignore_index=True
)

display(counts)
counts.to_csv(BASE / "component_counts_all.csv", index=False)

# Save wide tables
for dataset in DIRS:
    subset = counts[counts["Dataset"] == dataset]

    wide = subset.pivot(
        index=["Model", "Label"],
        columns="Section",
        values=["Correct", "Partial_Match", "Incorrect"]
    ).fillna(0).astype(int)

    wide = wide.reindex(MODEL_ORDER, level="Model")
    wide.to_csv(BASE / f"component_counts_{dataset}.csv")

    print(f"\n{dataset.upper()}")
    display(wide)

# Print TikZ coordinates
def tikz_coords(df, section, metric):
    values = df.set_index(["Model", "Section"])[metric]
    return "coordinates {\n" + "\n".join(
        f"  ({x:.2f},{int(values.get((model, section), 0))})"
        for x, model in zip(XPOS, MODEL_ORDER)
    ) + "\n};"

for dataset in DIRS:
    print(f"\n{'=' * 60}\n{dataset.upper()}\n{'=' * 60}")
    subset = counts[counts["Dataset"] == dataset]

    for section in SECTIONS:
        for metric in ["Correct", "Partial_Match", "Incorrect"]:
            print(f"\n{section} {metric}")
            print(tikz_coords(subset, section, metric))


## relationship loss analysis

In [ ]:

import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import Counter

# ============================================================
# REVIEWER 4, COMMENT 15 — FILTERING LOSS ANALYSIS
# ============================================================

BASE = ROOT / "CLOVER_output_files"
FINAL_DIR = BASE / "ctd_matching_combined_final"
SAVE_DIR = BASE / "filtering_loss_analysis"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "DeepSeek-E": "deepseek_E_output",
    "GPT-4.1-E": "gpt_E_output",
    "Claude-E": "claude_E_output",
    "GPT-4o-E": "gpt4o_E_output",
    "DS-V/GPT-4.1-E": "deepseekV_gptE_output",
    "DS-V/GPT-4o-E": "deepseekV_gpt4oE_output",
    "DS-V/Claude-E": "deepseekV_claudeE_output",
    "DS-V/DeepSeek-E": "deepseekV_deepseekE_output",
    "GPT-4.1-V/DeepSeek-E": "gptV_deepseekE_output",
    "GPT-4.1-V/Claude-E": "gptV_claudeE_output",
    "GPT-4.1-V/GPT-4.1-E": "gptV_gptE_output",
    "Cl-V/DeepSeek-E": "claudeV_deepseekE_output",
    "Cl-V/GPT-4.1-E": "claudeV_gptE_output",
    "Cl-V/GPT-4o-E": "claudeV_gpt4oE_output",
    "Cl-V/Claude-E": "claudeV_claudeE_output",
    "GPT-4o-V/DeepSeek-E": "gpt4oV_deepseekE_output",
    "GPT-4o-V/Claude-E": "gpt4oV_claudeE_output",
    "GPT-4o-V/GPT-4o-E": "gpt4oV_gpt4oE_output"
}

# These must match the fields used in your final CTD evaluator.
MATCH_FIELDS = [
    "pmid",
    "chemicalID",
    "geneSymbol",
    "relationship"
]

# These identify deduplicated prediction rows across stages.
# section_type is optional for the row comparison.
ROW_FIELDS = MATCH_FIELDS + ["section_type"]

# Set to True only if the final saved files agree with the
# reconstructed output. Nonmatching models will be skipped.
VERIFY_SAVED_FINAL = True

# ------------------------------------------------------------
# Canonical values
# ------------------------------------------------------------

def normalize_identifier(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    return re.sub(r"^([0-9]+)\.0+$", r"\1", s)


def normalize_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip().upper()


def normalize_relation(x):
    s = normalize_text(x)
    s = s.replace("^", " ")
    s = re.sub(r"[-_]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def canonical_series(df, col):
    if col not in df.columns:
        raise KeyError(f"Missing column: {col}")

    if col == "pmid":
        return df[col].map(normalize_identifier)

    if col == "relationship":
        return df[col].map(normalize_relation)

    if col == "chemicalID":
        return df[col].map(normalize_text)

    if col == "geneSymbol":
        return df[col].map(normalize_text)

    if col == "section_type":
        return df[col].map(normalize_text)

    return df[col].map(normalize_text)


def add_keys(df, fields, prefix):
    out = df.copy()

    for field in fields:
        if field == "section_type" and field not in out:
            out[field] = ""

        out[f"_{prefix}_{field}"] = canonical_series(
            out, field
        )

    out[f"_{prefix}_key"] = list(
        zip(*[
            out[f"_{prefix}_{field}"]
            for field in fields
        ])
    )

    return out


def key_counter(df, fields):
    temp = add_keys(df, fields, "check")
    return Counter(temp["_check_key"])


# ------------------------------------------------------------
# Identify CTD reference fields
# ------------------------------------------------------------

def resolve_column(df, possibilities, description):
    for name in possibilities:
        if name in df.columns:
            return name

    raise KeyError(
        f"Could not find {description}. "
        f"Available columns: {list(df.columns)}"
    )


def prepare_ctd_reference(manual):
    ref = manual.copy()

    column_map = {
        "pmid": resolve_column(
            ref,
            ["pmid", "PMID", "PubMedIDs", "PubMedID"],
            "CTD PMID"
        ),
        "chemicalID": resolve_column(
            ref,
            ["chemicalID", "ChemicalID", "ChemicalId"],
            "CTD chemical ID"
        ),
        "geneSymbol": resolve_column(
            ref,
            ["geneSymbol", "GeneSymbol", "# GeneSymbol"],
            "CTD gene symbol"
        ),
        "relationship": resolve_column(
            ref,
            [
                "relationship_manual",
                "relationship",
                "InteractionActions",
                "interactionActions"
            ],
            "CTD relationship"
        )
    }

    ref = ref.rename(
        columns={
            original: target
            for target, original in column_map.items()
            if original != target
        }
    )

    # An interaction must be atomic, e.g.
    # increases^expression, not a multi-action expression.
    # This routine does not split complex CTD actions.
    ref = add_keys(ref, MATCH_FIELDS, "ctd")

    ref_keys = set(ref["_ctd_key"])

    return ref_keys


CTD_KEYS = prepare_ctd_reference(ctd_ixns)

print(f"Unique CTD reference keys: {len(CTD_KEYS):,}")

# IMPORTANT:
# Verify that this CTD reference matches the exact
# single-step reference set used by your evaluator.
# In particular, if your evaluator expands multiple
# interaction actions, prepare that expansion before
# constructing CTD_KEYS.


# ------------------------------------------------------------
# Track prediction-level CTD agreement
# ------------------------------------------------------------

def add_match_status(df):
    out = add_keys(df, MATCH_FIELDS, "match")

    out["CTD_Exact_Match"] = out["_match_key"].isin(CTD_KEYS)

    out["CTD_Status"] = np.where(
        out["CTD_Exact_Match"],
        "CTD-matched",
        "CTD-unmatched"
    )

    return out


# ------------------------------------------------------------
# Find rows removed between stages
# Handles duplicate occurrences with occurrence counters.
# ------------------------------------------------------------

def find_removed_rows(before, after):

    before = add_keys(before, ROW_FIELDS, "row")
    after = add_keys(after, ROW_FIELDS, "row")

    before["_occurrence"] = (
        before.groupby("_row_key", sort=False).cumcount()
    )

    after["_occurrence"] = (
        after.groupby("_row_key", sort=False).cumcount()
    )

    after_members = set(
        zip(
            after["_row_key"],
            after["_occurrence"]
        )
    )

    before["_retained"] = [
        (key, n) in after_members
        for key, n in zip(
            before["_row_key"],
            before["_occurrence"]
        )
    ]

    removed = before.loc[
        ~before["_retained"]
    ].copy()

    retained = before.loc[
        before["_retained"]
    ].copy()

    return removed, retained


# ------------------------------------------------------------
# Text clues for manual review
#
# Diagnostic only; these are NOT necessarily the actual
# rules that caused filter_dataframe() to drop a row.
# ------------------------------------------------------------

REVIEW_PATTERNS = {
    "undetectable": r"\bundetectable\b|\bnot detectable\b",
    "below_detection": (
        r"\bbelow (?:the )?(?:limit|level) of detection\b"
        r"|\bbelow detection\b"
    ),
    "decrease": (
        r"\bdecreas(?:e|ed|es|ing)\b"
        r"|\breduc(?:e|ed|es|ing|tion)\b"
        r"|\blower(?:ed|s|ing)?\b"
        r"|\bdownregulat(?:e|ed|ion|ing)\b"
        r"|\binhibit(?:ed|ion|s|ing)?\b"
        r"|\bsuppress(?:ed|ion|es|ing)\b"
    ),
    "null_finding": (
        r"\bno (?:significant )?(?:effect|change|difference)\b"
        r"|\bnot significant\b"
        r"|\bdid not (?:alter|change|affect)\b"
        r"|\bunchanged\b"
    ),
    "prior_work": (
        r"\bprevious(?:ly)?\b"
        r"|\bprior studies\b"
        r"|\breported that\b"
        r"|\bet al\.\b"
    ),
    "uncertainty": (
        r"\bmight\b|\bmay\b|\bpossibly\b"
        r"|\bpotentially\b|\bsuggests?\b"
    )
}


def find_source_column(df):
    candidates = [
        "sourceText", "source_text", "SourceText",
        "source", "Source", "evidence",
        "Evidence", "explanation", "Explanation"
    ]

    return next(
        (c for c in candidates if c in df.columns),
        None
    )


def add_review_clues(df):
    out = df.copy()
    source_col = find_source_column(out)

    if source_col is None:
        out["Review_Source_Text"] = ""
    else:
        out["Review_Source_Text"] = (
            out[source_col].fillna("").astype(str)
        )

    for label, pattern in REVIEW_PATTERNS.items():
        out[f"clue_{label}"] = (
            out["Review_Source_Text"]
            .str.contains(
                pattern,
                case=False,
                regex=True,
                na=False
            )
        )

    out["Potential_Decrease_Loss"] = (
        out["clue_decrease"] |
        out["clue_undetectable"] |
        out["clue_below_detection"]
    )

    def combine_clues(row):
        return "; ".join([
            label
            for label in REVIEW_PATTERNS
            if row[f"clue_{label}"]
        ])

    out["Text_Clues"] = out.apply(
        combine_clues, axis=1
    )

    out["Manual_Decision"] = "NOT_REVIEWED"
    out["Manual_Notes"] = ""
    out["Likely_Filter_Reason"] = "NOT_ESTABLISHED"

    return out


# ------------------------------------------------------------
# Utility: check saved final output
# ------------------------------------------------------------

def verify_final(model, reconstructed):

    path = FINAL_DIR / f"{model}_filtered_predictions.csv"

    if not path.exists():
        print(f"Missing saved final file: {path}")
        return False

    saved = pd.read_csv(path, low_memory=False)

    # Check exact prediction multisets, with section if present.
    fields = MATCH_FIELDS.copy()

    if (
        "section_type" in reconstructed.columns
        and "section_type" in saved.columns
    ):
        fields.append("section_type")

    a = key_counter(reconstructed, fields)
    b = key_counter(saved, fields)

    missing = b - a
    extra = a - b

    print(
        f"Saved verification: "
        f"reconstructed={len(reconstructed):,}, "
        f"saved={len(saved):,}, "
        f"missing={sum(missing.values()):,}, "
        f"extra={sum(extra.values()):,}"
    )

    return not missing and not extra


# ------------------------------------------------------------
# Main analysis
# ------------------------------------------------------------

summary = []
all_removed = []
failed = []

for i, (label, model) in enumerate(MODELS.items(), 1):

    print("\n" + "=" * 70)
    print(f"[{i}/{len(MODELS)}] {label}")
    print("=" * 70)

    raw = load_merged(model)

    if raw is None:
        failed.append(model)
        continue

    # Stage 1: Preprocessing
    before = update_dataframe(raw).reset_index(drop=True)

    # Stage 2: External gene normalization
    external = normalize_gene_symbols(
        before.copy(),
        default_species="hsa",
        strict_conflicts=True,
        use_mirgenedb=True,
        use_hgnc=True,
        use_uniprot=True
    ).reset_index(drop=True)

    # Stage 3: CTD gene normalization
    standardized = update_gene_symbol_(
        external.copy(),
        ctd_genes,
        ctd_lookup=CTD_SYMBOL_LOOKUP
    ).reset_index(drop=True)

    # Stage 4: Deduplication
    standardized = drop_duplicates_(
        standardized
    ).copy()

    # Stage 5: Apply original filtering unchanged
    filtered = filter_dataframe(
        standardized.copy(),
        ctd_chemicals,
        ctd_genes
    ).copy()

    # IMPORTANT: verify before reporting the counts
    if VERIFY_SAVED_FINAL:
        if not verify_final(model, filtered):
            print("SKIPPED: final reconstruction mismatch")
            failed.append(model)
            continue

    # Add CTD status to both stages
    pre = add_match_status(standardized)
    post = add_match_status(filtered)

    # Track removed predictions
    removed, retained = find_removed_rows(pre, post)

    removed = add_review_clues(removed)

    removed.insert(0, "Model", label)
    removed.insert(1, "Model_Key", model)

    # Aggregate model-level figures
    pre_tp = int(pre["CTD_Exact_Match"].sum())
    post_tp = int(post["CTD_Exact_Match"].sum())

    removed_tp = int(
        removed["CTD_Exact_Match"].sum()
    )

    removed_unmatched = int(
        (~removed["CTD_Exact_Match"]).sum()
    )

    pre_count = len(pre)
    post_count = len(post)

    print(f"Pre-filter rows: {pre_count:,}")
    print(f"Post-filter rows: {post_count:,}")
    print(f"Removed rows: {len(removed):,}")
    print(f"Removed CTD-matched: {removed_tp:,}")
    print(f"Removed CTD-unmatched: {removed_unmatched:,}")

    # This is prediction-row recall only if your evaluator
    # counts one TP per unique CTD reference key.
    # Use your original evaluator for final P/R metrics.
    summary.append({
        "Model": label,
        "Pre_Filter_Rows": pre_count,
        "Post_Filter_Rows": post_count,
        "Removed_Rows": len(removed),
        "Pre_Filter_CTD_Matched_Rows": pre_tp,
        "Post_Filter_CTD_Matched_Rows": post_tp,
        "Removed_CTD_Matched_Rows": removed_tp,
        "Removed_CTD_Unmatched_Rows": removed_unmatched,
        "Removed_With_Decrease_Clues": int(
            removed["Potential_Decrease_Loss"].sum()
        ),
        "Removed_With_Undetectable_Clues": int(
            removed["clue_undetectable"].sum()
        ),
        "Removed_With_Null_Clues": int(
            removed["clue_null_finding"].sum()
        ),
        "Removal_Percent": (
            100 * len(removed) / pre_count
            if pre_count else 0
        )
    })

    all_removed.append(removed)

    # Save model-specific removed predictions
    removed.to_csv(
        SAVE_DIR / f"{model}_removed_predictions.csv",
        index=False
    )

    # High-priority manual review
    priority = removed.loc[
        removed["CTD_Exact_Match"] |
        removed["Potential_Decrease_Loss"]
    ].copy()

    priority.to_csv(
        SAVE_DIR / f"{model}_priority_manual_review.csv",
        index=False
    )


# ------------------------------------------------------------
# Combined outputs
# ------------------------------------------------------------

summary_df = pd.DataFrame(summary)

removed_df = (
    pd.concat(all_removed, ignore_index=True)
    if all_removed
    else pd.DataFrame()
)

summary_df.to_csv(
    SAVE_DIR / "filtering_loss_summary.csv",
    index=False
)

removed_df.to_csv(
    SAVE_DIR / "all_removed_predictions.csv",
    index=False
)

if not removed_df.empty:

    priority_df = removed_df.loc[
        removed_df["CTD_Exact_Match"] |
        removed_df["Potential_Decrease_Loss"]
    ].copy()

    priority_df.to_csv(
        SAVE_DIR / "all_priority_manual_review.csv",
        index=False
    )

    # Short review sheet with main evidence fields
    review_cols = [
        "Model",
        "pmid",
        "chemicalID",
        "chemicalName",
        "geneSymbol",
        "relationship",
        "section_type",
        "CTD_Status",
        "Review_Source_Text",
        "Text_Clues",
        "Likely_Filter_Reason",
        "Manual_Decision",
        "Manual_Notes"
    ]

    review_cols = [
        col for col in review_cols
        if col in priority_df.columns
    ]

    priority_df[review_cols].to_csv(
        SAVE_DIR / "manual_review_sheet.csv",
        index=False
    )


# ------------------------------------------------------------
# Overall results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("REVIEWER 4 — COMMENT 15: SUMMARY")
print("=" * 70)

display(summary_df)

if not summary_df.empty:

    print("\nTOTALS ACROSS VERIFIED CONFIGURATIONS")
    print(
        "Pre-filter predictions:",
        f"{summary_df['Pre_Filter_Rows'].sum():,}"
    )
    print(
        "Removed predictions:",
        f"{summary_df['Removed_Rows'].sum():,}"
    )
    print(
        "Removed CTD-matched rows:",
        f"{summary_df['Removed_CTD_Matched_Rows'].sum():,}"
    )
    print(
        "Removed CTD-unmatched rows:",
        f"{summary_df['Removed_CTD_Unmatched_Rows'].sum():,}"
    )
    print(
        "Removed rows with decrease-related wording:",
        f"{summary_df['Removed_With_Decrease_Clues'].sum():,}"
    )

    print(
        "\nNOTE: These totals pool model configurations, "
        "so the same biological interaction may appear "
        "multiple times across models."
    )

if failed:
    print("\nModels not verified:")
    for model in failed:
        print(" -", model)

print("\nOutput directory:", SAVE_DIR)
